# Train & Export Calibrated 2-Ply + Q3 Chess Value CNN (`nojs/chess` · v2.1 Drop-In)

This notebook trains a **plug-and-play `weights.json` upgrade** for `chess/` **without changing a single layer, channel count, kernel size, or JSON key** of `SVGChessValueCNN` (`2,623` parameters · `6 → 18 (5×5) → 27 (3×3) → 36 (2×2) + 3-ch skip_mat → 39→1` · `nojs-svg-chess-value-cnn-v1`, `arch_version: 2`).

### Four Training Upgrades Eliminating Piece Hangs, Missed Threats & Opening Mistakes

1. **Per-Triplet Mean-Centered Value Head + `3.8×` Finer 8-Bit Resolution (`M: 23.65 → ~6.25`)**:
   - Absorbs per-triplet feature means $\mu_g$ into each existing `value_head.color_matrices[g]` 5th-column offset (`0` extra SVG `<filter>` primitives), eliminating mid-loop accumulator drift.
   - Shrinks `logit_half_range_M` from `23.65` (`58 cp/LSB`, `74%` dead saturated LUT entries) down to **`~6.25` (`~15 cp/LSB`, `>95%` active 8-bit LUT utilization)** so 1-pawn (`-100 cp`), exchange (`-195 cp`), minor-piece (`-305 cp`), and opening (`25–60 cp`) differences span crisp multi-LSB margins instead of colliding onto identical bytes.
2. **Exact `2-Ply + Ply-3 Quiescence` Alignment & Turn-Flip Supervision**:
   - Embeds the exact browser search pipeline (`16-quadrant Ply-2 MVV-LVA` + `s2q0..s2q7` **Ply-3 Quiescence** + `s2m` checkmate override + `tie_ramp`).
   - Explicitly trains on **post-Ply-3 Quiescence leaves (`s0 → m1 → m2 → q3_ai`)** where `Opponent` moves next at Ply 4 while tensors remain encoded from `AI`'s perspective (`AI = 255, Opp = 1`). Whenever an AI Queen, Rook, Bishop, or Knight is left hanging on `s2_q` after an unrelated Ply-3 pawn capture, Stockfish labels `s2_q` at its true `-300` to `-950 cp` loss instead of assuming AI still has a tempo to retreat.
3. **Stage-2 Search-In-The-Loop Hard-Negative Mining (`DAgger`)**:
   - After Stage-1 8-bit QAT converges, runs the **actual quantized `2-Ply + Ply-3 Quiescence` SVG engine** across `3,000` opening & tactical positions, compares its move against Stockfish, harvests every blunder (`>= 55 cp` drop: hung Queens/Rooks/Minors, missed opponent next-ply captures/forks, opening traps) into exact `(best_minimax_leaf, blunder_minimax_leaf)` pairs, and fine-tunes for `12` QAT epochs.
4. **Anchored Orthogonal `skip_mat` + `48`-Stem Opening Discipline & Full-Hierarchy Monotonicity**:
   - Locks `skip_mat` Row 0 to exact FIDE piece ratios (`P=1.00, N=3.05, B=3.25, R=5.00, Q=9.50, K=0.00` — preventing `Rook` from ever drifting down toward `Bishop`), anchors Row 1 to Major-vs-Minor exchange conservation (`+Q, +R` vs `-N, -B`), and enforces counterfactual logit-drop margins across all piece hangs (`-Q, -R, -B, -N, -P`), bad trades (`QxR, QxB, QxN, RxB, RxN`), and pawn-attacked major-piece squares.

**Usage in Google Colab**: Select a **T4 GPU** runtime and click **Runtime $\rightarrow$ Run all** (~9–12 minutes end-to-end) to automatically download the drop-in `weights.json`.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Install Stockfish Binary & python-chess on Google Colab
# ---------------------------------------------------------------------------
import os
import platform
import shutil
import stat
import subprocess
import sys
import tarfile
import urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chess"])


def ensure_stockfish() -> str:
    """Locate or install a fast Stockfish UCI binary on Colab / Linux / macOS."""
    for candidate in (shutil.which("stockfish"), "/usr/games/stockfish", "/usr/local/bin/stockfish", "./stockfish_bin"):
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate

    # 1. Fast Ubuntu/Debian apt install on Google Colab (~4s)
    if shutil.which("apt-get"):
        print("Installing Stockfish via apt-get...")
        res = subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode == 0:
            subprocess.run(["apt-get", "install", "-y", "-qq", "stockfish"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for candidate in (shutil.which("stockfish"), "/usr/games/stockfish"):
            if candidate and os.path.isfile(candidate):
                return candidate

    # 2. Direct official GitHub release binary fallback for Linux x86_64
    if platform.system() == "Linux" and platform.machine() in ("x86_64", "AMD64"):
        print("Downloading official Stockfish Linux x86-64 AVX2 release...")
        url = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar"
        tar_path = "stockfish_avx2.tar"
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, "r") as tf:
            for member in tf.getmembers():
                if member.isfile() and "stockfish" in os.path.basename(member.name) and not member.name.endswith(".tar"):
                    extracted = tf.extractfile(member)
                    if extracted is not None:
                        with open("./stockfish_bin", "wb") as out_f:
                            out_f.write(extracted.read())
                        os.chmod("./stockfish_bin", os.stat("./stockfish_bin").st_mode | stat.S_IEXEC)
                        return "./stockfish_bin"

    raise RuntimeError("Could not find or install Stockfish binary!")


STOCKFISH_PATH = ensure_stockfish()
print(f"Stockfish ready at: {STOCKFISH_PATH}")


In [ ]:
# ---------------------------------------------------------------------------
# 1. Imports, Constants & Bit-Exact HTML5 SVG `2-Ply + Ply-3 Quiescence` Engine
# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import json
import math
import random
import time
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import chess
import chess.engine
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Exact 8-bit integer zero-point: 128 / 255 avoids half-integer rounding drift!
ZERO_POINT_U8 = 128
ZERO_POINT = round(ZERO_POINT_U8 / 255.0, 6)  # 0.501961
POS_VAL_U8 = 255                              # +127/255 above Z0 (AI piece)
NEG_VAL_U8 = 1                                # -127/255 below Z0 (Opponent piece)
CP_WDL_SCALE = 310.0                          # Crisp centipawn -> WDL logistic scale

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {device} | CPU count: {os.cpu_count()} | Z0 = {ZERO_POINT} ({ZERO_POINT_U8}/255)")

PIECE_TYPES = (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING)
PIECE_VAL = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
PIECE_CP = {chess.PAWN: 100, chess.KNIGHT: 305, chess.BISHOP: 325, chess.ROOK: 500, chess.QUEEN: 950, chess.KING: 0}
LVA_ORDER = {chess.PAWN: 1, chess.KNIGHT: 2, chess.BISHOP: 3, chess.ROOK: 4, chess.QUEEN: 5, chess.KING: 6}
MVV_TIER = {chess.PAWN: 1, chess.KNIGHT: 2, chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 4}

AI_RAYS: Tuple[Tuple[int, int, int], ...] = (
    (1, 0, 0), (-1, 0, 0),
    (0, 1, 1), (0, -1, 1),
    (1, 1, 2), (-1, -1, 2),
    (1, -1, 3), (-1, 1, 3),
)
AI_KNIGHT_DELTAS: Tuple[Tuple[int, int], ...] = (
    (-1, -2), (1, -2), (-2, -1), (2, -1),
    (-2, 1), (2, 1), (-1, 2), (1, 2),
)
AI_KING_DELTAS: Tuple[Tuple[int, int], ...] = (
    (-1, -1), (0, -1), (1, -1),
    (-1, 0),           (1, 0),
    (-1, 1),  (0, 1),  (1, 1),
)


def encode_board_uint8(board: chess.Board, ai_color: Optional[chess.Color] = None) -> np.ndarray:
    """Encode `board` into uint8 `(6, 8, 8)` from `ai_color`'s perspective (defaults to `board.turn`)."""
    persp = board.turn if ai_color is None else ai_color
    arr = np.full((6, 8, 8), ZERO_POINT_U8, dtype=np.uint8)
    for sq, pc in board.piece_map().items():
        r, f = divmod(sq, 8)
        ar = r if persp == chess.WHITE else (7 - r)
        arr[pc.piece_type - 1, ar, f] = POS_VAL_U8 if pc.color == persp else NEG_VAL_U8
    return arr


def norm_sq_to_real(sq_norm: int, ai_color: chess.Color) -> int:
    ny, nx = divmod(sq_norm, 8)
    ry = ny if ai_color == chess.WHITE else (7 - ny)
    return 8 * ry + nx


def real_sq_to_norm(sq_real: int, ai_color: chess.Color) -> int:
    ry, rx = divmod(sq_real, 8)
    ny = ry if ai_color == chess.WHITE else (7 - ry)
    return 8 * ny + rx


def compute_tie_ramp_u8() -> np.ndarray:
    """Exact 64-entry destination tie-breaker ramp matching `svg_ai.py` (`s5a`/`s5b`)."""
    scored = []
    for a in range(64):
        ar, af = divmod(a, 8)
        centrality = -2.2 * abs(af - 3.5) - 1.0 * abs(ar - 3.3)
        scored.append((centrality, a))
    scored.sort(key=lambda item: (item[0], -item[1]))
    ramp = np.zeros(64, dtype=np.uint8)
    for rank_idx, (_c, a) in enumerate(scored):
        ramp[a] = 128 + rank_idx
    return ramp


TIE_RAMP_U8 = compute_tie_ramp_u8()


def compute_ply2_base_tier(pt_src: int, mvv: int, is_r0: int) -> int:
    if mvv > 0:
        if pt_src == 1 and is_r0:
            return 31 if mvv == 4 else (25 + mvv)
        base = 25 if mvv == 4 else (6 * mvv)
        return base + (6 - pt_src)
    if pt_src == 1 and is_r0:
        return 24
    return {6: 0, 1: 1, 5: 2, 4: 3, 3: 4, 2: 5}[pt_src]


def ply3_tier_and_placed(pt3: int, tc: int) -> Tuple[int, int]:
    """Bit-exact mirror of `svg_ai.ply3_tier_and_placed` (`s2q0..s2q7` SEE-filtered Ply-3 Quiescence).
    Note: when `def_opp == 1` and `gain > 0`, `placed_code == 0` folds Opponent's Ply-4 recapture!
    """
    if tc == 0:
        return 0, 0
    if 172 <= tc <= 175:
        if pt3 != 6:
            return 0, 0
        mvv = (tc - 172) + 1
        return {1: 2, 2: 5, 3: 8, 4: 12}[mvv], 6
    if not (140 <= tc <= 171):
        return 0, 0
    idx = tc - 140
    mvv = (idx // 8) + 1
    rem = idx % 8
    def_opp = rem // 4
    is_r7 = (rem // 2) % 2
    is_chk1 = rem % 2

    if def_opp == 0:
        if pt3 == 1 and is_r7 == 1:
            return (15 if mvv == 4 else 14), 5
        hi = pt3 in (5, 6)
        tier_map = {
            1: 2 if hi else 3,
            2: 5 if hi else 6,
            3: 8 if hi else 9,
            4: 12 if hi else 13,
        }
        return tier_map[mvv], pt3

    if pt3 == 6:
        return 0, 0
    val_m = {1: 1, 2: 3, 3: 5, 4: 9}[mvv]
    val_a = {1: 1, 2: 3, 3: 3, 4: 5, 5: 9}[pt3]
    gain = val_m - val_a
    if gain <= -1:
        return 0, 0
    if gain == 0:
        return (1, 0) if is_chk1 == 1 else (0, 0)
    return {2: 4, 4: 7, 6: 10, 8: 11}[gain], 0


def _shift_8x8(arr: np.ndarray, dx: int, dy: int) -> np.ndarray:
    out = np.zeros_like(arr)
    x0_s, x1_s = max(0, -dx), min(8, 8 - dx)
    y0_s, y1_s = max(0, -dy), min(8, 8 - dy)
    out[y0_s + dy : y1_s + dy, x0_s + dx : x1_s + dx] = arr[y0_s:y1_s, x0_s:x1_s]
    return out


def _ray_prop_8x8(src: np.ndarray, empty: np.ndarray, dx: int, dy: int) -> np.ndarray:
    r = _shift_8x8(src, dx, dy)
    e = _shift_8x8(empty, dx, dy)
    for mul in (1, 2):
        rs = _shift_8x8(r, mul * dx, mul * dy)
        es = _shift_8x8(e, mul * dx, mul * dy)
        r = np.clip(r + e * rs, 0, 1)
        e = e * es
    rs = _shift_8x8(r, 4 * dx, 4 * dy)
    return np.clip(r + e * rs, 0, 1)


def select_and_apply_ply3_quiescence_8x8(s2_u8: np.ndarray) -> Tuple[np.ndarray, Optional[Tuple[int, int, int, int]]]:
    """Bit-exact mirror of `svg_ai._select_and_apply_ply3_quiescence_8x8` (`s2q0..s2q7`)."""
    ai = (s2_u8 == 255).astype(np.int32)
    op = (s2_u8 <= 1).astype(np.int32)
    ai_P, ai_N, ai_B, ai_R, ai_Q, ai_K = [ai[i] for i in range(6)]
    op_P, op_N, op_B, op_R, op_Q, op_K = [op[i] for i in range(6)]
    ai_any = np.clip(ai.sum(axis=0), 0, 1)
    op_any = np.clip(op.sum(axis=0), 0, 1)
    empty = 1 - np.clip(ai_any + op_any, 0, 1)

    opp_att = np.clip(_shift_8x8(op_P, 1, -1) + _shift_8x8(op_P, -1, -1), 0, 1)
    for dx, dy in AI_KNIGHT_DELTAS:
        opp_att = np.clip(opp_att + _shift_8x8(op_N, dx, dy), 0, 1)
    for dx, dy in AI_KING_DELTAS:
        opp_att = np.clip(opp_att + _shift_8x8(op_K, dx, dy), 0, 1)

    chk_pn_p = op_P * np.clip(_shift_8x8(ai_K, -1, 1) + _shift_8x8(ai_K, 1, 1), 0, 1)
    kn_k = np.zeros((8, 8), dtype=np.int32)
    for dx, dy in AI_KNIGHT_DELTAS:
        kn_k = np.clip(kn_k + _shift_8x8(ai_K, dx, dy), 0, 1)
    chk_sq = np.clip(chk_pn_p + op_N * kn_k, 0, 1)

    pin_ax = np.zeros((8, 8), dtype=np.int32)
    axes = (
        (1, (1, 0), np.clip(op_R + op_Q, 0, 1)),
        (2, (0, 1), np.clip(op_R + op_Q, 0, 1)),
        (3, (1, 1), np.clip(op_B + op_Q, 0, 1)),
        (4, (1, -1), np.clip(op_B + op_Q, 0, 1)),
    )
    for ax_code, (dx, dy), op_sl in axes:
        rp_r = _ray_prop_8x8(op_sl, empty, dx, dy)
        rp_g = _ray_prop_8x8(ai_K, empty, dx, dy)
        rn_r = _ray_prop_8x8(op_sl, empty, -dx, -dy)
        rn_g = _ray_prop_8x8(ai_K, empty, -dx, -dy)
        xray_p = _shift_8x8(ai_K * rp_r, dx, dy)
        xray_n = _shift_8x8(ai_K * rn_r, -dx, -dy)
        opp_att = np.clip(opp_att + rp_r + rn_r + xray_p + xray_n, 0, 1)
        chk_sq = np.clip(chk_sq + np.clip(rp_g + rn_g, 0, 1) * op_sl, 0, 1)
        line_ax = np.clip(rp_g * rn_r + rn_g * rp_r, 0, 1)
        pin_ax = pin_ax + ax_code * line_ax

    n_chk = int(chk_sq.sum())
    ai_nonk = ai_any * (1 - ai_K)
    src_state = np.zeros((8, 8), dtype=np.int32)
    for y in range(8):
        for x in range(8):
            if not ai_nonk[y, x]:
                continue
            if n_chk >= 2:
                src_state[y, x] = 130
            elif n_chk == 1:
                src_state[y, x] = 131 if pin_ax[y, x] == 0 else 130
            else:
                src_state[y, x] = 131 if pin_ax[y, x] == 0 else (131 + int(pin_ax[y, x]))

    mvv_grid = op_P * 1 + np.clip(op_N + op_B, 0, 1) * 2 + op_R * 3 + op_Q * 4
    tgt_code = np.zeros((8, 8), dtype=np.int32)
    for y in range(8):
        for x in range(8):
            mvv = int(mvv_grid[y, x])
            if mvv == 0:
                continue
            def_opp = 1 if opp_att[y, x] > 0 else 0
            is_r7 = 1 if y == 7 else 0
            if n_chk == 0:
                tgt_code[y, x] = 140 + 8 * (mvv - 1) + 4 * def_opp + 2 * is_r7
            elif n_chk == 1 and chk_sq[y, x] == 1:
                tgt_code[y, x] = 140 + 8 * (mvv - 1) + 4 * def_opp + 2 * is_r7 + 1
            elif def_opp == 0:
                tgt_code[y, x] = 172 + (mvv - 1)

    def _cand(u3: int, vx: int, vy: int, pt3: int) -> Tuple[int, int, int]:
        if u3 < 0:
            return (0, 0, 0)
        tier, placed = ply3_tier_and_placed(pt3, int(tgt_code[vy, vx]))
        if tier == 0:
            return (0, 0, 0)
        return (128 + 8 * (tier - 1) + vy, 128 + u3, 128 + 8 * placed + vx)

    def _mux(a: Tuple[int, int, int], b: Tuple[int, int, int]) -> Tuple[int, int, int]:
        return b if b[0] > a[0] else a

    grid_cand = [[(0, 0, 0) for _ in range(8)] for _ in range(8)]
    for vy in range(8):
        for vx in range(8):
            if tgt_code[vy, vx] == 0:
                continue
            orth_best = 0
            for dx, dy, allow_st in ((1, 0, (131, 132)), (-1, 0, (131, 132)), (0, 1, (131, 133)), (0, -1, (131, 133))):
                ux, uy = vx - dx, vy - dy
                while 0 <= ux < 8 and 0 <= uy < 8:
                    if empty[uy, ux]:
                        ux -= dx; uy -= dy
                        continue
                    if int(src_state[uy, ux]) in allow_st:
                        u3 = 8 * uy + ux
                        if ai_R[uy, ux]:
                            orth_best = max(orth_best, 192 + u3)
                        elif ai_Q[uy, ux]:
                            orth_best = max(orth_best, 128 + u3)
                    break

            diag_best = 0
            for dx, dy, allow_st in ((1, 1, (131, 134)), (-1, -1, (131, 134)), (1, -1, (131, 135)), (-1, 1, (131, 135))):
                ux, uy = vx - dx, vy - dy
                while 0 <= ux < 8 and 0 <= uy < 8:
                    if empty[uy, ux]:
                        ux -= dx; uy -= dy
                        continue
                    if int(src_state[uy, ux]) in allow_st:
                        u3 = 8 * uy + ux
                        if ai_B[uy, ux]:
                            diag_best = max(diag_best, 192 + u3)
                        elif ai_Q[uy, ux]:
                            diag_best = max(diag_best, 128 + u3)
                    break

            u_b = (diag_best - 192) if diag_best >= 192 else -1
            u_r = (orth_best - 192) if orth_best >= 192 else -1
            q_o = (orth_best - 128) if (128 <= orth_best < 192) else -1
            q_d = (diag_best - 128) if (128 <= diag_best < 192) else -1
            u_q = max(q_o, q_d)

            u_k = -1
            for dx, dy in AI_KING_DELTAS:
                ux, uy = vx - dx, vy - dy
                if 0 <= ux < 8 and 0 <= uy < 8 and ai_K[uy, ux]:
                    u_k = 8 * uy + ux
                    break

            slk = _mux(_mux(_mux(_cand(u_b, vx, vy, 3), _cand(u_r, vx, vy, 4)), _cand(u_q, vx, vy, 5)), _cand(u_k, vx, vy, 6))

            u_p = -1
            for dx, allow_st in ((1, (131, 134)), (-1, (131, 135))):
                ux, uy = vx - dx, vy - 1
                if 0 <= ux < 8 and 0 <= uy < 8 and ai_P[uy, ux] and int(src_state[uy, ux]) in allow_st:
                    u_p = max(u_p, 8 * uy + ux)

            u_n = -1
            for dx, dy in AI_KNIGHT_DELTAS:
                ux, uy = vx - dx, vy - dy
                if 0 <= ux < 8 and 0 <= uy < 8 and ai_N[uy, ux] and int(src_state[uy, ux]) == 131:
                    u_n = max(u_n, 8 * uy + ux)

            pn = _mux(_cand(u_p, vx, vy, 1), _cand(u_n, vx, vy, 2))
            grid_cand[vy][vx] = _mux(pn, slk)

    cur = [[grid_cand[y][x] for x in range(8)] for y in range(8)]
    for step in (1, 2, 4):
        nxt = [[(0, 0, 0) for _ in range(8)] for _ in range(8)]
        for y in range(8):
            for x in range(8):
                b = cur[y][x + step] if x + step < 8 else (0, 0, 0)
                nxt[y][x] = _mux(cur[y][x], b)
        cur = nxt
    for step in (1, 2, 4):
        nxt = [[(0, 0, 0) for _ in range(8)] for _ in range(8)]
        for y in range(8):
            for x in range(8):
                b = cur[y + step][x] if y + step < 8 else (0, 0, 0)
                nxt[y][x] = _mux(cur[y][x], b)
        cur = nxt

    win_r, win_g, win_b = cur[0][0]
    if win_r < 128:
        return s2_u8.copy(), None

    v3y = (win_r - 128) % 8
    u3 = win_g - 128
    placed_code = (win_b - 128) // 8
    v3x = (win_b - 128) % 8
    v3 = 8 * v3y + v3x
    u3y, u3x = divmod(u3, 8)

    out = s2_u8.copy()
    out[:, u3y, u3x] = ZERO_POINT_U8
    out[:, v3y, v3x] = ZERO_POINT_U8
    if 1 <= placed_code <= 6:
        out[placed_code - 1, v3y, v3x] = POS_VAL_U8
    return out, (u3, v3, placed_code, (win_r - 128) // 8 + 1)


def apply_q3_to_pychess_board(b2: chess.Board, ai_col: chess.Color) -> Tuple[chess.Board, np.ndarray, bool]:
    """Apply exact SVG Ply-3 Quiescence (`s2q0..s2q7`) to `b2`, returning `(b_leaf, s2_q_u8, q3_fired)`.
    - When `placed_code in 1..6` (undefended capture), `b_leaf = b3` (`Opponent` to move at Ply 4!).
    - When `placed_code == 0` (defended winning trade, SEE Ply-4 recapture folded!), also pushes Opponent's
      recapture onto `v3_real` so `b_leaf` matches `s2_q_u8` (`AI` to move at Ply 5!) for Stockfish labeling!
    """
    s2_u8 = encode_board_uint8(b2, ai_color=ai_col)
    if b2.is_checkmate():
        return b2, s2_u8, False
    s2_q, q3 = select_and_apply_ply3_quiescence_8x8(s2_u8)
    if q3 is None:
        return b2, s2_u8, False
    u3_n, v3_n, placed_code, _tier = q3
    ur = norm_sq_to_real(u3_n, ai_col)
    vr = norm_sq_to_real(v3_n, ai_col)
    pc = b2.piece_at(ur)
    promo = chess.QUEEN if (pc is not None and pc.piece_type == chess.PAWN and chess.square_rank(vr) in (0, 7)) else None
    mv3 = chess.Move(ur, vr, promotion=promo)
    if mv3 not in b2.legal_moves:
        return b2, s2_u8, False
    b3 = b2.copy(stack=False)
    b3.push(mv3)
    if placed_code == 0 and not b3.is_game_over():
        recaps = [m for m in b3.legal_moves if m.to_square == vr and m.promotion in (None, chess.QUEEN)]
        if recaps:
            recaps.sort(key=lambda m: PIECE_VAL.get(b3.piece_type_at(m.from_square) or chess.QUEEN, 9))
            b4 = b3.copy(stack=False)
            b4.push(recaps[0])
            return b4, s2_q, True
    return b3, s2_q, True


def _ply1_css_priority_key(board: chess.Board, mv: chess.Move) -> Tuple[int, int]:
    """Exact `.e10/.e11` CSS Ply-1 move bus priority (`css_chess.py` lines 1098-1244):
      - Captures / En Passant: `Pawn (0) < Knight (1) < Bishop (2) < Rook (3) < Queen (4) < King (6)`
      - Quiet moves:           `Knight (1) < Bishop (2) < Rook (3) < Queen (4) < Pawn (5) < King (6)`
    """
    pt = board.piece_type_at(mv.from_square)
    is_cap = board.is_capture(mv)
    if pt == chess.PAWN:
        grp = 0 if is_cap else 5
        sub = abs(mv.to_square - mv.from_square)
    elif pt == chess.KNIGHT:
        grp = 1
        sub = mv.from_square
    elif pt in (chess.BISHOP, chess.ROOK, chess.QUEEN):
        grp = {chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4}[pt]
        dr = (mv.from_square // 8) - (mv.to_square // 8)
        df = (mv.from_square % 8) - (mv.to_square % 8)
        sgn_r = 0 if dr == 0 else (1 if dr > 0 else -1)
        sgn_f = 0 if df == 0 else (1 if df > 0 else -1)
        ray_order = {(0, -1): 0, (0, 1): 1, (-1, 0): 2, (1, 0): 3, (-1, -1): 4, (1, 1): 5, (-1, 1): 6, (1, -1): 7}
        sub = ray_order.get((sgn_r, sgn_f), 8)
    else:
        grp = 6
        sub = 99 if board.is_castling(mv) else mv.from_square
    return (grp, sub)


def select_ply2_quadrant_moves_pychess(b1: chess.Board, ai_col: chess.Color) -> List[chess.Move]:
    """Exact 16-quadrant Ply-2 reply selector matching `svg_ai._select_ply2_quadrant_replies` (`Tier 12` `ai_k_adj`)."""
    ai_ksq = b1.king(ai_col)
    quad_best: Dict[Tuple[int, int], Tuple[int, int, chess.Move]] = {}
    for m2 in b1.legal_moves:
        if m2.promotion not in (None, chess.QUEEN) or b1.is_castling(m2) or b1.is_en_passant(m2):
            continue
        v2_n = real_sq_to_norm(m2.to_square, ai_col)
        u2_n = real_sq_to_norm(m2.from_square, ai_col)
        ty, tx = divmod(v2_n, 8)
        vic = b1.piece_type_at(m2.to_square)
        mvv = MVV_TIER[vic] if vic is not None else 0
        pt_s = LVA_ORDER[b1.piece_type_at(m2.from_square)]
        bt = compute_ply2_base_tier(pt_s, mvv, 1 if ty == 0 else 0)
        if pt_s == 5 and ai_ksq is not None and chess.square_distance(m2.to_square, ai_ksq) == 1:
            bt = max(bt, 12)
        sc2 = 128 + 4 * bt + 2 * (ty % 2) + (tx % 2)
        qk = (ty // 2, tx // 2)
        prev = quad_best.get(qk)
        if prev is None or sc2 > prev[0] or (sc2 == prev[0] and u2_n > prev[1]):
            quad_best[qk] = (sc2, u2_n, m2)
    return [val[2] for val in quad_best.values()]


def select_move_2ply_q3_svg_batch(
    boards: Sequence[chess.Board],
    eval_u8_fn: Callable[[np.ndarray], np.ndarray],
) -> List[Tuple[chess.Move, int, Dict[str, int], Dict[str, chess.Board]]]:
    """Vectorized multi-board `2-Ply + Ply-3 Quiescence` SVG Minimax search!
    Evaluates all leaves across `boards` in batched calls to `eval_u8_fn(leaves_u8_n688) -> scores_u8`,
    returning `(best_move, best_q_u8, uci_to_q_u8, uci_to_minimax_leaf_board)` per root board.
    """
    all_leaves_u8: List[np.ndarray] = []
    leaf_meta: List[Tuple[int, int, chess.Board]] = []
    per_board_m1: List[Dict[int, chess.Move]] = []
    per_board_q: List[Dict[int, int]] = []
    per_board_s1_chk: List[Dict[int, bool]] = []
    per_board_worst_leaf: List[Dict[int, chess.Board]] = []

    for b_idx, board in enumerate(boards):
        ai_col = board.turn
        m1_by_vnorm: Dict[int, Tuple[Tuple[int, int], chess.Move]] = {}
        for mv in board.legal_moves:
            if mv.promotion not in (None, chess.QUEEN):
                continue
            vn = real_sq_to_norm(mv.to_square, ai_col)
            pkey = _ply1_css_priority_key(board, mv)
            prev = m1_by_vnorm.get(vn)
            if prev is None or pkey < prev[0]:
                m1_by_vnorm[vn] = (pkey, mv)

        m1_map = {vn: tup[1] for vn, tup in m1_by_vnorm.items()}
        q_map = {vn: 255 for vn in m1_map}
        chk_map: Dict[int, bool] = {}
        wleaf_map: Dict[int, chess.Board] = {}

        for vn, m1 in m1_map.items():
            b1 = board.copy(stack=False)
            b1.push(m1)
            chk_map[vn] = b1.is_check()
            wleaf_map[vn] = b1

            m2_list = select_ply2_quadrant_moves_pychess(b1, ai_col)
            if not m2_list:
                q_map[vn] = 254 if chk_map[vn] else ZERO_POINT_U8
                continue

            for m2 in m2_list:
                b2 = b1.copy(stack=False)
                b2.push(m2)
                if b2.is_checkmate():
                    if 2 < q_map[vn]:
                        q_map[vn] = 2
                        wleaf_map[vn] = b2
                    continue
                b3, s2_q_u8, _ = apply_q3_to_pychess_board(b2, ai_col)
                all_leaves_u8.append(s2_q_u8)
                leaf_meta.append((b_idx, vn, b3))

        per_board_m1.append(m1_map)
        per_board_q.append(q_map)
        per_board_s1_chk.append(chk_map)
        per_board_worst_leaf.append(wleaf_map)

    if all_leaves_u8:
        scores_all = eval_u8_fn(np.stack(all_leaves_u8, axis=0))
        for idx, (b_idx, vn, b3) in enumerate(leaf_meta):
            val = int(scores_all[idx])
            if val < per_board_q[b_idx][vn]:
                per_board_q[b_idx][vn] = val
                per_board_worst_leaf[b_idx][vn] = b3

    results = []
    for b_idx, board in enumerate(boards):
        m1_map = per_board_m1[b_idx]
        q_map = per_board_q[b_idx]
        chk_map = per_board_s1_chk[b_idx]
        wleaf_map = per_board_worst_leaf[b_idx]
        for vn in q_map:
            if q_map[vn] == 255:
                q_map[vn] = 254 if chk_map.get(vn, False) else ZERO_POINT_U8
        best_vn = max(q_map.keys(), key=lambda vn: (q_map[vn], int(TIE_RAMP_U8[vn])))
        uci_scores = {m1_map[vn].uci(): q_map[vn] for vn in m1_map}
        uci_leaves = {m1_map[vn].uci(): wleaf_map[vn] for vn in m1_map}
        results.append((m1_map[best_vn], q_map[best_vn], uci_scores, uci_leaves))
    return results


In [ ]:
# ---------------------------------------------------------------------------
# 2. Stage-1 Corpus Generator: 48 Opening Stems, Post-Ply-3 Turn-Flip Leaves,
#    All-Piece Hangs (`Q/R/B/N/P`), Unrescued Threats & Bad Exchanges (~196k Instances)
# ---------------------------------------------------------------------------

# 48 Classical ECO & Common Human Rematch Opening Stems
OPENING_BOOK_UCI: List[List[str]] = [
    [],
    ["e2e4"], ["d2d4"], ["c2c4"], ["g1f3"],
    ["e2e4", "e7e5", "g1f3"],                                             # 2.Nf3 attacking e5!
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "f8c5"],                     # Giuoco Piano / Italian
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "f8c5", "c2c3", "g8f6"],     # Italian Mainline
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6"],                     # Two Knights Defense
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6", "f3g5"],             # Fried Liver / Knight Attack on f7!
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6", "f3g5", "d7d5", "e4d5"],  # Two Knights Ulvestad/Morphy
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6", "d2d3", "f8c5"],     # Giuoco Pianissimo
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5", "a7a6", "b5a4", "g8f6"],     # Ruy Lopez Morphy Defense
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5", "g8f6", "e1g1"],             # Ruy Lopez Berlin
    ["e2e4", "e7e5", "g1f3", "b8c6", "d2d4", "e5d4", "f3d4"],             # Scotch Game
    ["e2e4", "e7e5", "g1f3", "b8c6", "b1c3", "g8f6", "f1b5"],             # Four Knights Spanish
    ["e2e4", "e7e5", "g1f3", "g8f6", "f3e5", "d7d6", "e5f3", "f6e4"],     # Petrov Defense
    ["e2e4", "e7e5", "g1f3", "d7d6", "d2d4", "e5d4", "f3d4"],             # Philidor Defense
    ["e2e4", "e7e5", "f1c4", "g8f6", "d2d3"],                             # Bishop's Opening
    ["e2e4", "e7e5", "f1c4", "b8c6", "d1h5", "g7g6", "h5f3", "g8f6"],     # Scholar's Mate Defense
    ["e2e4", "e7e5", "d1h5", "b8c6", "f1c4", "g7g6", "h5f3", "g8f6"],     # Wayward Queen Attack
    ["e2e4", "e7e5", "g1f3", "d8h4", "b1c3"],                             # Premature Black Qh4 refutation
    ["e2e4", "e7e5", "g1f3", "d8f6", "b1c3"],                             # Premature Black Qf6 refutation
    ["e2e4", "c7c5", "g1f3", "d7d6", "d2d4", "c5d4", "f3d4", "g8f6", "b1c3"],  # Open Sicilian Najdorf/Dragon
    ["e2e4", "c7c5", "g1f3", "b8c6", "d2d4", "c5d4", "f3d4", "g8f6"],     # Classical Sicilian
    ["e2e4", "c7c5", "g1f3", "e7e6", "d2d4", "c5d4", "f3d4"],             # Sicilian Taimanov/Kan
    ["e2e4", "c7c5", "g1f3", "b8c6", "f1b5"],                             # Rossolimo Sicilian
    ["e2e4", "c7c5", "c2c3", "g8f6", "e4e5", "f6d5"],                     # Alapin Sicilian
    ["e2e4", "c7c5", "b1c3", "b8c6", "g1f3", "d7d6"],                     # Closed Sicilian
    ["e2e4", "e7e6", "d2d4", "d7d5", "b1c3", "g8f6"],                     # Classical French
    ["e2e4", "e7e6", "d2d4", "d7d5", "e4e5", "c7c5", "c2c3"],             # Advance French
    ["e2e4", "c7c6", "d2d4", "d7d5", "b1c3", "d5e4", "c3e4"],             # Classical Caro-Kann
    ["e2e4", "c7c6", "d2d4", "d7d5", "e4e5", "c8f5"],                     # Advance Caro-Kann
    ["e2e4", "d7d5", "e4d5", "d8d5", "b1c3", "d5a5", "d2d4", "g8f6"],     # Scandinavian Qa5
    ["e2e4", "d7d5", "e4d5", "d8d5", "b1c3", "d5d8"],                     # Scandinavian Qd8 retreat
    ["e2e4", "d7d6", "d2d4", "g8f6", "b1c3", "g7g6"],                     # Pirc Defense
    ["d2d4", "d7d5", "c2c4", "e7e6", "b1c3", "g8f6", "c1g5"],             # Queen's Gambit Declined
    ["d2d4", "d7d5", "c2c4", "c7c6", "g1f3", "g8f6", "b1c3"],             # Slav Defense
    ["d2d4", "d7d5", "c2c4", "d5c4", "g1f3", "g8f6", "e2e3"],             # Queen's Gambit Accepted
    ["d2d4", "d7d5", "f1f4", "g8f6", "e2e3", "c7c5", "c2c3"],             # London System
    ["d2d4", "g8f6", "f1f4", "d7d5", "e2e3", "e7e6", "g1f3"],             # London vs Nf6
    ["d2d4", "g8f6", "c2c4", "g7g6", "b1c3", "f8g7", "e2e4", "d7d6"],     # King's Indian Defense
    ["d2d4", "g8f6", "c2c4", "e7e6", "b1c3", "f8b4", "e2e3"],             # Nimzo-Indian Defense
    ["d2d4", "g8f6", "c2c4", "e7e6", "g1f3", "b7b6"],                     # Queen's Indian Defense
    ["d2d4", "g8f6", "c2c4", "g7g6", "b1c3", "d7d5"],                     # Grunfeld Defense
    ["c2c4", "e7e5", "b1c3", "g8f6", "g2g3", "d7d5"],                     # English Opening
    ["g1f3", "d7d5", "g2g3", "g8f6", "f1g2", "c7c6", "e1g1"],             # Reti / King's Indian Attack
    ["e2e4", "e7e5", "f2f4", "e5f4", "g1f3", "d7d5"],                     # King's Gambit Modern Defense
]

# Instance tuple: `(fen, ai_is_white, regime)`
BoardInstance = Tuple[str, bool, str]
PairInstance = Tuple[Tuple[str, bool], Tuple[str, bool]]


def cp_to_wdl(cp: float, scale: float = CP_WDL_SCALE) -> float:
    x = max(-18.0, min(18.0, cp / scale))
    w = 1.0 / (1.0 + math.exp(-x))
    return float(min(0.995, max(0.005, w)))


def _find_blunders_and_unrescued_threats(board: chess.Board) -> List[Tuple[chess.Move, int, int]]:
    """Find legal Ply-1 moves that either:
      1. Move a piece (`Q, R, B, N, P`) onto an attacked square with net material loss >= 1 (`target_sq = mv.to_square`), OR
      2. Leave an ALREADY-attacked AI piece (`Q, R, B, N, P`) hanging en prise on `sq` while playing an unrelated move (`target_sq = sq`)!
    Returns `(bad_move, priority, victim_sq_for_opp)` sorted worst-first.
    """
    ai_col = board.turn
    opp = not ai_col
    cands: List[Tuple[chess.Move, int, int]] = []

    # Pre-identify currently attacked AI pieces on `board`
    existing_threat_sqs = [
        sq for sq, pc in board.piece_map().items()
        if pc.color == ai_col and pc.piece_type != chess.KING and board.is_attacked_by(opp, sq)
    ]

    for mv in board.legal_moves:
        if mv.promotion not in (None, chess.QUEEN):
            continue
        p_att = board.piece_type_at(mv.from_square)
        if p_att is None:
            continue
        v_att = PIECE_VAL[p_att]
        p_vic = board.piece_type_at(mv.to_square)
        v_vic = PIECE_VAL.get(p_vic, 0) if p_vic is not None else 0

        b1 = board.copy(stack=False)
        b1.push(mv)
        if b1.is_checkmate():
            continue

        # Case 1: Moved piece lands on an Opponent-attacked square (`QxN`, `RxN`, or direct hang)
        if p_att != chess.KING and mv.promotion is None and b1.is_attacked_by(opp, mv.to_square):
            # Minimum value of Opponent attacker on `mv.to_square`
            opp_atts = [b1.piece_type_at(s) for s in b1.attackers(opp, mv.to_square)]
            min_opp_v = min((PIECE_VAL.get(pt, 9) for pt in opp_atts if pt != chess.KING), default=0)
            is_defended = b1.is_attacked_by(ai_col, mv.to_square)
            net_loss = (v_att - v_vic) if not is_defended else (v_att - v_vic - min_opp_v)
            if net_loss >= 1:
                prio = 2 * net_loss + (6 if p_att == chess.QUEEN else (3 if p_att == chess.ROOK else 0))
                cands.append((mv, prio, mv.to_square))
                continue

        # Case 2: Unrescued existing threat (`AI` ignores an attacked Q/R/B/N/P on `sq`!)
        for sq in existing_threat_sqs:
            if sq == mv.from_square:
                continue
            pc_hang = b1.piece_at(sq)
            if pc_hang is None or pc_hang.color != ai_col:
                continue
            if b1.is_attacked_by(opp, sq):
                v_h = PIECE_VAL[pc_hang.piece_type]
                opp_atts = [b1.piece_type_at(s) for s in b1.attackers(opp, sq)]
                min_opp_v = min((PIECE_VAL.get(pt, 9) for pt in opp_atts if pt != chess.KING), default=0)
                is_def = b1.is_attacked_by(ai_col, sq)
                loss_est = (v_h - v_vic) if not is_def else (v_h - v_vic - min_opp_v)
                if loss_est >= 1:
                    prio = 2 * loss_est + (6 if pc_hang.piece_type == chess.QUEEN else 2)
                    cands.append((mv, prio, sq))
                    break

    cands.sort(key=lambda t: t[1], reverse=True)
    return cands


def _best_ply2_punish(b1: chess.Board, target_sq: int, ai_col: chess.Color, eng: chess.engine.SimpleEngine, lim: chess.engine.Limit) -> Optional[chess.Move]:
    """Select Opponent's Ply-2 punishing move (preferring quadrant-legal LVA capture on `target_sq` or Stockfish's top reply)."""
    leg = [m for m in b1.legal_moves if m.promotion in (None, chess.QUEEN)]
    if not leg:
        return None
    recaps = [m for m in leg if m.to_square == target_sq]
    if recaps:
        recaps.sort(key=lambda m: PIECE_VAL.get(b1.piece_type_at(m.from_square) or chess.QUEEN, 9))
        return recaps[0]
    res = eng.play(b1, lim)
    return res.move if (res.move in leg) else leg[0]


def _advance_to_svg_leaf(b1: chess.Board, m2: chess.Move, ai_col: chess.Color, apply_q3: bool = True) -> Tuple[str, bool]:
    """Push Opponent Ply-2 `m2` and optional exact SVG Ply-3 Quiescence (`s2q0..s2q7`), returning `(fen, ai_is_white)`.
    Crucially, when Ply-3 Quiescence fires, `b3.turn == opponent` while `ai_is_white == (ai_col == WHITE)`!
    """
    b2 = b1.copy(stack=False)
    b2.push(m2)
    if apply_q3 and not b2.is_game_over():
        b3, _, _ = apply_q3_to_pychess_board(b2, ai_col)
        return b3.fen(), (ai_col == chess.WHITE)
    return b2.fen(), (ai_col == chess.WHITE)


def _synthesize_endgame_or_imbalance_board(rng: random.Random) -> Optional[chess.Board]:
    templates = [
        ([chess.QUEEN], []),
        ([chess.ROOK], []),
        ([chess.QUEEN, chess.PAWN, chess.PAWN], [chess.KNIGHT, chess.PAWN, chess.PAWN]),
        ([chess.QUEEN, chess.KNIGHT, chess.PAWN], [chess.BISHOP, chess.KNIGHT, chess.PAWN]),
        ([chess.QUEEN, chess.PAWN], [chess.ROOK, chess.PAWN]),
        ([chess.QUEEN, chess.PAWN], [chess.ROOK, chess.KNIGHT]),
        ([chess.ROOK, chess.PAWN, chess.PAWN], [chess.KNIGHT, chess.PAWN, chess.PAWN]),
        ([chess.ROOK, chess.PAWN], [chess.BISHOP, chess.PAWN]),
        ([chess.BISHOP, chess.KNIGHT, chess.PAWN], [chess.ROOK]),
        ([chess.KNIGHT, chess.PAWN, chess.PAWN], [chess.PAWN, chess.PAWN]),
        ([chess.BISHOP, chess.PAWN, chess.PAWN], [chess.PAWN, chess.PAWN]),
        ([chess.PAWN, chess.PAWN], [chess.PAWN]),
    ]
    strong_pcs, weak_pcs = rng.choice(templates)
    w_pcs, b_pcs = (strong_pcs, weak_pcs) if rng.random() < 0.5 else (weak_pcs, strong_pcs)

    for _ in range(30):
        b = chess.Board.empty()
        sqs = rng.sample(range(64), 2 + len(w_pcs) + len(b_pcs))
        wk, bk = sqs[0], sqs[1]
        if chess.square_distance(wk, bk) <= 1:
            continue
        b.set_piece_at(wk, chess.Piece(chess.KING, chess.WHITE))
        b.set_piece_at(bk, chess.Piece(chess.KING, chess.BLACK))
        ok, idx = True, 2
        for col, plist in ((chess.WHITE, w_pcs), (chess.BLACK, b_pcs)):
            for pt in plist:
                sq = sqs[idx]; idx += 1
                if pt == chess.PAWN and chess.square_rank(sq) in (0, 7):
                    ok = False; break
                b.set_piece_at(sq, chess.Piece(pt, col))
            if not ok:
                break
        if ok and b.is_valid():
            b.turn = chess.WHITE if rng.random() < 0.5 else chess.BLACK
            if b.is_valid():
                return b
    return None


def generate_stage1_corpus(
    sf_path: str,
    target_singletons: int = 120000,
    target_pairs: int = 38000,
    seed: int = SEED,
) -> Tuple[List[BoardInstance], List[PairInstance]]:
    """Generate Stage-1 training singletons & contrastive leaf pairs aligned with `2-Ply + Ply-3 Quiescence`!"""
    num_workers = max(2, min(6, (os.cpu_count() or 2)))
    rng_main = random.Random(seed)
    seen_keys = set()
    singletons: List[BoardInstance] = []
    contrast_pairs: List[PairInstance] = []

    def _inst_key(fen: str, ai_w: bool) -> str:
        p = fen.split(" ")
        return f"{p[0]} {p[1]} {p[2]} {p[3]} {1 if ai_w else 0}"

    def _try_add(fen: str, ai_w: bool, reg: str) -> None:
        if len(singletons) < target_singletons:
            k = _inst_key(fen, ai_w)
            if k not in seen_keys:
                seen_keys.add(k)
                singletons.append((fen, ai_w, reg))

    # 1. Clean endgames & material-imbalance boards (~13% of singletons)
    n_end_target = int(0.13 * target_singletons)
    while len(singletons) < n_end_target:
        eb = _synthesize_endgame_or_imbalance_board(rng_main)
        if eb is not None:
            _try_add(eb.fen(), eb.turn == chess.WHITE, "endgame")

    # 2. Dedicated High-Depth Opening Contrastive Pairs (~8,000 pairs across all 48 stems + 1-ply perturbations!)
    opening_roots: List[str] = []
    seen_op = set()
    for stem in OPENING_BOOK_UCI:
        b_op = chess.Board()
        for step_i in range(len(stem) + 1):
            fk = b_op.board_fen() + (" w" if b_op.turn == chess.WHITE else " b")
            if fk not in seen_op and not b_op.is_game_over():
                seen_op.add(fk)
                opening_roots.append(b_op.fen())
            # Also include 1-ply human deviations (early Queen moves, flank pawns, Knight jumps) from each book node!
            for alt_mv in list(b_op.legal_moves):
                b_alt = b_op.copy(stack=False)
                b_alt.push(alt_mv)
                afk = b_alt.board_fen() + (" w" if b_alt.turn == chess.WHITE else " b")
                if afk not in seen_op and not b_alt.is_game_over() and len(opening_roots) < 2600:
                    seen_op.add(afk)
                    opening_roots.append(b_alt.fen())
            if step_i < len(stem):
                mv_s = chess.Move.from_uci(stem[step_i])
                if mv_s in b_op.legal_moves:
                    b_op.push(mv_s)

    rem_sing_per_worker = math.ceil((target_singletons - len(singletons)) / num_workers) + 2000
    pairs_per_worker = math.ceil(target_pairs / num_workers) + 500
    op_chunks = [opening_roots[i::num_workers] for i in range(num_workers)]

    def _rollout_worker(w_id: int) -> Tuple[List[BoardInstance], List[PairInstance]]:
        w_rng = random.Random(seed + 1009 * (w_id + 1))
        w_singles: List[BoardInstance] = []
        w_pairs: List[PairInstance] = []
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 24})
            lim_fast = chess.engine.Limit(nodes=500)
            lim_mpv = chess.engine.Limit(nodes=750)
            lim_open = chess.engine.Limit(nodes=2600)

            # First: harvest crisp high-depth Opening Discipline pairs on `op_chunks[w_id]`!
            for op_fen in op_chunks[w_id]:
                b = chess.Board(op_fen)
                ai_col = b.turn
                leg = [m for m in b.legal_moves if m.promotion in (None, chess.QUEEN)]
                if len(leg) < 4:
                    continue
                infos = eng.analyse(b, lim_open, multipv=min(8, len(leg)))
                pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"] and inf["pv"][0] in leg]
                if not pvs:
                    continue
                m_best = pvs[0]
                bg = b.copy(stack=False); bg.push(m_best)
                if bg.is_game_over():
                    continue
                rg = eng.play(bg, lim_fast).move or next(iter(bg.legal_moves))
                leaf_good = _advance_to_svg_leaf(bg, rg, ai_col, apply_q3=True)
                w_singles.append((leaf_good[0], leaf_good[1], "opening_q3"))

                # Contrast `m_best` against: (a) premature Queen sorties, (b) opening blunders/hangs, (c) flank/weak moves
                bad_cands: List[chess.Move] = []
                q_sorties = [m for m in leg if b.piece_type_at(m.from_square) == chess.QUEEN and m != m_best]
                flank_pawns = [
                    m for m in leg
                    if b.piece_type_at(m.from_square) == chess.PAWN
                    and chess.square_file(m.from_square) in (0, 5, 6, 7)
                    and m not in pvs[:2]
                ]
                blunders = [t[0] for t in _find_blunders_and_unrescued_threats(b)[:2] if t[0] != m_best]
                bad_cands.extend(blunders)
                if q_sorties:
                    bad_cands.append(w_rng.choice(q_sorties))
                if flank_pawns:
                    bad_cands.append(w_rng.choice(flank_pawns))
                if len(pvs) >= 3:
                    bad_cands.append(pvs[-1])

                used_b = set()
                for m_bad in bad_cands[:3]:
                    if m_bad == m_best or m_bad in used_b:
                        continue
                    used_b.add(m_bad)
                    bb = b.copy(stack=False); bb.push(m_bad)
                    if bb.is_game_over():
                        continue
                    rb = eng.play(bb, lim_fast).move or next(iter(bb.legal_moves))
                    leaf_bad = _advance_to_svg_leaf(bb, rb, ai_col, apply_q3=True)
                    w_pairs.append((leaf_good, leaf_bad))
                    w_singles.append((leaf_bad[0], leaf_bad[1], "opening_bad"))

            # Second: game rollouts harvesting post-Ply-3 Turn-Flip hangs, unrescued threats, & positional pairs
            while len(w_singles) < rem_sing_per_worker or len(w_pairs) < pairs_per_worker:
                b = chess.Board()
                if w_rng.random() < 0.80:
                    stem = w_rng.choice(OPENING_BOOK_UCI)
                    cut = w_rng.randint(0, len(stem)) if stem else 0
                    for uci in stem[:cut]:
                        mv_b = chess.Move.from_uci(uci)
                        if mv_b in b.legal_moves:
                            b.push(mv_b)

                max_plies = w_rng.randint(24, 88)
                for ply in range(max_plies):
                    if b.is_game_over():
                        w_singles.append((b.fen(), b.turn == chess.WHITE, "terminal"))
                        break

                    leg = [m for m in b.legal_moves if m.promotion in (None, chess.QUEEN)]
                    if not leg:
                        break
                    ai_col = b.turn

                    # --- A. Harvest Unsound-Trade / Piece-Hang / Unrescued-Threat Pairs (with exact Ply-3 Quiescence!) ---
                    blunders = _find_blunders_and_unrescued_threats(b)
                    if blunders and len(w_pairs) < pairs_per_worker and w_rng.random() < 0.48:
                        m1_good = eng.play(b, lim_fast).move
                        if m1_good not in leg:
                            m1_good = leg[0]
                        m1_bad, _prio, vic_sq = blunders[0] if w_rng.random() < 0.72 else w_rng.choice(blunders[:4])
                        if m1_good != m1_bad:
                            bg = b.copy(stack=False); bg.push(m1_good)
                            bb = b.copy(stack=False); bb.push(m1_bad)
                            if not bg.is_game_over() and not bb.is_game_over():
                                rg = eng.play(bg, lim_fast).move or next(iter(bg.legal_moves))
                                rb = _best_ply2_punish(bb, vic_sq, ai_col, eng, lim_fast)
                                if rb is not None:
                                    use_q3 = (w_rng.random() < 0.82)
                                    leaf_g = _advance_to_svg_leaf(bg, rg, ai_col, apply_q3=use_q3)
                                    leaf_b = _advance_to_svg_leaf(bb, rb, ai_col, apply_q3=use_q3)
                                    w_pairs.append((leaf_g, leaf_b))
                                    w_singles.append((leaf_b[0], leaf_b[1], "hang_or_q3"))

                    # --- B. Harvest Turn-Flip Distraction Traps (AI piece stays attacked on `s2_q` after Ply-3 capture!) ---
                    if len(w_singles) < rem_sing_per_worker and w_rng.random() < 0.22:
                        m_try = w_rng.choice(leg)
                        bt1 = b.copy(stack=False); bt1.push(m_try)
                        if not bt1.is_game_over():
                            m2_list = select_ply2_quadrant_moves_pychess(bt1, ai_col)
                            if m2_list:
                                m2_pick = w_rng.choice(m2_list)
                                bt2 = bt1.copy(stack=False); bt2.push(m2_pick)
                                if not bt2.is_game_over():
                                    bt3, _, q3_fired = apply_q3_to_pychess_board(bt2, ai_col)
                                    if q3_fired:
                                        w_singles.append((bt3.fen(), ai_col == chess.WHITE, "q3_turnflip"))

                    # --- C. Harvest Positional Good-vs-Bad Pairs (with Ply-3 Quiescence!) ---
                    elif len(leg) >= 5 and len(w_pairs) < pairs_per_worker and w_rng.random() < 0.22:
                        infos = eng.analyse(b, lim_mpv, multipv=min(6, len(leg)))
                        pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"] and inf["pv"][0] in leg]
                        if len(pvs) >= 2:
                            m1_best = pvs[0]
                            quiets = [m for m in leg if not b.is_capture(m) and m != m1_best]
                            m1_bad_pos = pvs[-1] if (w_rng.random() < 0.65 or not quiets) else w_rng.choice(quiets)
                            if m1_best != m1_bad_pos:
                                bg = b.copy(stack=False); bg.push(m1_best)
                                bb = b.copy(stack=False); bb.push(m1_bad_pos)
                                if not bg.is_game_over() and not bb.is_game_over():
                                    rg = eng.play(bg, lim_fast).move or next(iter(bg.legal_moves))
                                    rb = eng.play(bb, lim_fast).move or next(iter(bb.legal_moves))
                                    leaf_g = _advance_to_svg_leaf(bg, rg, ai_col, apply_q3=True)
                                    leaf_b = _advance_to_svg_leaf(bb, rb, ai_col, apply_q3=True)
                                    w_pairs.append((leaf_g, leaf_b))
                                    w_singles.append((leaf_b[0], leaf_b[1], "bad_pos"))

                    # --- D. Advance Rollout ---
                    roll = w_rng.random()
                    if roll < 0.60:
                        if w_rng.random() < 0.70 or len(leg) < 3:
                            mv = eng.play(b, lim_fast).move or w_rng.choice(leg)
                        else:
                            infos = eng.analyse(b, lim_fast, multipv=min(3, len(leg)))
                            pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"]]
                            mv = w_rng.choice(pvs) if pvs else w_rng.choice(leg)
                        used_tag = "opening" if ply < 16 else ("endgame" if len(b.piece_map()) <= 10 else "sf_play")
                    elif roll < 0.76 and len(leg) >= 4:
                        infos = eng.analyse(b, lim_fast, multipv=min(6, len(leg)))
                        pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"]]
                        mv = pvs[-1] if len(pvs) >= 2 else w_rng.choice(leg)
                        used_tag = "bad_pos"
                    elif roll < 0.88 and blunders:
                        mv = blunders[0][0]
                        used_tag = "hang_or_q3"
                    else:
                        mv = w_rng.choice(leg)
                        used_tag = "random_mv"

                    if mv not in b.legal_moves:
                        mv = w_rng.choice(leg)
                    b.push(mv)
                    if w_rng.random() < 0.45 or used_tag in ("opening", "bad_pos", "hang_or_q3"):
                        w_singles.append((b.fen(), b.turn == chess.WHITE, used_tag))
        finally:
            eng.quit()
        return w_singles, w_pairs

    t0 = time.time()
    print(f"Generating Stage-1 Stockfish + Opening + Ply-3 Quiescence corpus across {num_workers} UCI workers...")
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        results = list(pool.map(_rollout_worker, range(num_workers)))

    for w_singles, w_pairs in results:
        for fen, ai_w, reg in w_singles:
            _try_add(fen, ai_w, reg)
        for pair in w_pairs:
            if len(contrast_pairs) < target_pairs:
                contrast_pairs.append(pair)

    rng_main.shuffle(contrast_pairs)
    contrast_pairs = contrast_pairs[:target_pairs]
    n_turnflip = sum(1 for f, aw, _ in singletons if (f.split(" ")[1] == "w") != aw)
    print(
        f"Generated {len(singletons):,} singletons ({n_turnflip:,} post-Ply-3 turn-flip leaves!) + "
        f"{len(contrast_pairs):,} contrastive pairs ({len(singletons) + 2 * len(contrast_pairs):,} total instances) in {time.time() - t0:.1f}s"
    )
    regs = [r for _, _, r in singletons]
    for reg in sorted(set(regs)):
        cnt = regs.count(reg)
        print(f"  Regime {reg:13s}: {cnt:6,d} ({100.0 * cnt / len(regs):4.1f}%)")
    return singletons, contrast_pairs


SINGLETONS, CONTRAST_PAIRS = generate_stage1_corpus(
    STOCKFISH_PATH,
    target_singletons=120000,
    target_pairs=38000,
)


In [ ]:
# ---------------------------------------------------------------------------
# 3. Deduplicated Parallel Persistent-UCI Stockfish Labeling (Exact `ai_color` POV!)
# ---------------------------------------------------------------------------

SF_NODES_LIMIT = 2200


def label_instances_with_stockfish(
    instances: Sequence[Tuple[str, bool]],
    sf_path: str,
    nodes_limit: int = SF_NODES_LIMIT,
) -> Tuple[np.ndarray, np.ndarray]:
    """Deduplicate `instances` by `(board_fen, turn, castling, ep)` so Stockfish evaluates each unique FEN once,
    then compute exact `(encoded_u8, wdl_f32)` from each slot's `ai_is_white` perspective — properly scoring both
    `AI-to-move` leaves AND post-Ply-3 Quiescence `Opponent-to-move` leaves!
    """
    n_total = len(instances)
    unique_fens: List[str] = []
    key_to_uidx: Dict[str, int] = {}
    slot_to_uidx = np.empty(n_total, dtype=np.int32)
    slot_ai_white = np.array([aw for _, aw in instances], dtype=bool)

    for i, (fen, _aw) in enumerate(instances):
        parts = fen.split(" ")
        key = f"{parts[0]} {parts[1]} {parts[2]} {parts[3]}"
        uidx = key_to_uidx.get(key)
        if uidx is None:
            uidx = len(unique_fens)
            key_to_uidx[key] = uidx
            unique_fens.append(fen)
        slot_to_uidx[i] = uidx

    n_uniq = len(unique_fens)
    u_white_u8 = np.empty((n_uniq, 6, 8, 8), dtype=np.uint8)
    u_black_u8 = np.empty((n_uniq, 6, 8, 8), dtype=np.uint8)
    u_wdl_white = np.empty(n_uniq, dtype=np.float32)

    num_workers = max(2, min(6, (os.cpu_count() or 2)))
    chunk_size = math.ceil(n_uniq / num_workers)

    def _worker(w_id: int, start: int, end: int) -> None:
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 32})
            lim = chess.engine.Limit(nodes=nodes_limit)
            for idx in range(start, end):
                bd = chess.Board(unique_fens[idx])
                u_white_u8[idx] = encode_board_uint8(bd, ai_color=chess.WHITE)
                u_black_u8[idx] = encode_board_uint8(bd, ai_color=chess.BLACK)
                if bd.is_checkmate():
                    # `bd.turn` is checkmated -> White lost if `bd.turn == WHITE`, else White won!
                    u_wdl_white[idx] = 0.005 if bd.turn == chess.WHITE else 0.995
                elif bd.is_stalemate() or bd.is_insufficient_material():
                    u_wdl_white[idx] = 0.500
                else:
                    info = eng.analyse(bd, lim)
                    pov_w = info["score"].pov(chess.WHITE)
                    mate_n = pov_w.mate()
                    if mate_n is not None:
                        u_wdl_white[idx] = 0.995 if mate_n > 0 else 0.005
                    else:
                        cp_w = float(pov_w.score() or 0.0)
                        u_wdl_white[idx] = cp_to_wdl(cp_w)
                if w_id == 0 and ((idx - start + 1) % 6000 == 0):
                    pct = 100.0 * (idx - start + 1) / max(1, end - start)
                    print(f"  [Worker 0] Labeled {idx - start + 1:,}/{end - start:,} ({pct:.0f}%)...")
        finally:
            eng.quit()

    t0 = time.time()
    print(f"Labeling {n_uniq:,} unique boards ({n_total:,} total instances) with Stockfish ({nodes_limit} nodes/pos)...")
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futs = []
        for w_id in range(num_workers):
            s_i = w_id * chunk_size
            e_i = min(n_uniq, (w_id + 1) * chunk_size)
            if s_i < e_i:
                futs.append(pool.submit(_worker, w_id, s_i, e_i))
        for fut in futs:
            fut.result()

    elapsed = time.time() - t0
    print(f"Stockfish labeled {n_uniq:,} unique boards in {elapsed:.1f}s ({n_uniq / max(elapsed, 1e-3):,.0f} pos/sec across {num_workers} UCI workers)")

    out_x = np.where(slot_ai_white[:, None, None, None], u_white_u8[slot_to_uidx], u_black_u8[slot_to_uidx])
    w_sc = u_wdl_white[slot_to_uidx]
    out_y = np.where(slot_ai_white, w_sc, (1.0 - w_sc).astype(np.float32))
    return out_x, out_y.astype(np.float32)


sing_insts = [(f, aw) for f, aw, _ in SINGLETONS]
pair_insts = [inst for pair in CONTRAST_PAIRS for inst in pair]
all_insts = sing_insts + pair_insts

x_all_u8, y_all_wdl = label_instances_with_stockfish(all_insts, STOCKFISH_PATH, nodes_limit=SF_NODES_LIMIT)

n_sing = len(SINGLETONS)
x_sing_u8, y_sing_wdl = x_all_u8[:n_sing], y_all_wdl[:n_sing]
x_pairs_u8 = x_all_u8[n_sing:].reshape(len(CONTRAST_PAIRS), 2, 6, 8, 8)
y_pairs_wdl = y_all_wdl[n_sing:].reshape(len(CONTRAST_PAIRS), 2)

rng_np = np.random.default_rng(SEED)
perm = rng_np.permutation(n_sing)
n_train = int(0.84 * n_sing)
n_val = int(0.08 * n_sing)
tr_idx, va_idx, te_idx = perm[:n_train], perm[n_train : n_train + n_val], perm[n_train + n_val :]

x_train_f = x_sing_u8[tr_idx].astype(np.float32) / 255.0
y_train_f = y_sing_wdl[tr_idx]
x_val_f = x_sing_u8[va_idx].astype(np.float32) / 255.0
y_val_f = y_sing_wdl[va_idx]
x_test_f = x_sing_u8[te_idx].astype(np.float32) / 255.0
y_test_f = y_sing_wdl[te_idx]

fens_test = [SINGLETONS[i][0] for i in te_idx]
ai_white_test = [SINGLETONS[i][1] for i in te_idx]
regimes_test = [SINGLETONS[i][2] for i in te_idx]
is_stm_test = [(f.split(" ")[1] == "w") == aw for f, aw in zip(fens_test, ai_white_test)]

n_pair_tr = int(0.90 * len(CONTRAST_PAIRS))
xp_train_f = x_pairs_u8[:n_pair_tr].astype(np.float32) / 255.0
yp_train_f = y_pairs_wdl[:n_pair_tr]
xp_val_f = x_pairs_u8[n_pair_tr:].astype(np.float32) / 255.0
yp_val_f = y_pairs_wdl[n_pair_tr:]

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train_f), torch.from_numpy(y_train_f)), batch_size=BATCH_SIZE, shuffle=True)
pair_loader = DataLoader(TensorDataset(torch.from_numpy(xp_train_f), torch.from_numpy(yp_train_f)), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val_f), torch.from_numpy(y_val_f)), batch_size=512, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test_f), torch.from_numpy(y_test_f)), batch_size=512, shuffle=False)

plt.figure(figsize=(7.8, 2.8))
plt.hist(y_train_f, bins=50, color="#38bdf8", edgecolor="#0f172a", alpha=0.85)
plt.axvline(0.5, color="#fbbf24", linestyle="--", label="Equal (0.0 cp)")
plt.title(f"Stockfish Distilled Target WDL ({len(y_train_f):,} Train Singletons + {n_pair_tr:,} 2-Ply+Q3 Pairs)")
plt.xlabel("Target Win Probability (AI Perspective)")
plt.ylabel("Positions")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 4. Identical-Architecture `SVGChessValueCNN` (`2,623` Params, `0` BatchNorm)
#    with Anchored `skip_mat` & Per-Triplet Mean-Centered Value Head (`M ≈ 6.25`)
# ---------------------------------------------------------------------------

def ste_q8(x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
    """Exact SVG intermediate RGBA8888 buffer physics: clamp to [0, 1] + 8-bit uint8 STE rounding."""
    xc = torch.clamp(x, 0.0, 1.0)
    if not quantize:
        return xc
    xq = torch.round(xc * 255.0) / 255.0
    return xc + (xq - xc).detach()


class SVGPointwise1x1(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, init_gain: float = 1.0, center_input: bool = False):
        super().__init__()
        assert in_channels % 3 == 0 and out_channels % 3 == 0
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        nn.init.kaiming_normal_(self.weight, mode="fan_in", nonlinearity="relu")
        with torch.no_grad():
            self.weight.mul_(init_gain)
            if center_input:
                self.bias.copy_(0.06 - self.weight.sum(dim=1) * ZERO_POINT)
            else:
                self.bias.fill_(0.03)

    def compute_channel_scales(self) -> torch.Tensor:
        with torch.no_grad():
            w_pos = torch.clamp(self.weight, min=0.0).sum(dim=1)
            w_neg = torch.clamp(-self.weight, min=0.0).sum(dim=1)
            return torch.clamp(torch.maximum(w_pos / 0.44, w_neg / 0.45), min=1.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        g_in = self.in_channels // 3
        z0 = ZERO_POINT
        s_ch = self.compute_channel_scales()
        s_col = s_ch.unsqueeze(1)
        s_view = s_ch.view(1, self.out_channels, 1, 1)

        w0 = (self.weight[:, 0:3] / s_col).view(self.out_channels, 3, 1, 1)
        acc = ste_q8(z0 + F.conv2d(x[:, 0:3], w0), quantize=quantize)
        for g in range(1, g_in):
            wg = (self.weight[:, 3 * g : 3 * (g + 1)] / s_col).view(self.out_channels, 3, 1, 1)
            part = ste_q8(z0 + F.conv2d(x[:, 3 * g : 3 * (g + 1)], wg), quantize=quantize)
            acc = ste_q8(acc + part - z0, quantize=quantize)

        b_view = self.bias.view(1, self.out_channels, 1, 1)
        return ste_q8(s_view * acc + (b_view - z0 * s_view), quantize=quantize)


class SVGLinearSkip6to3(nn.Module):
    """Anchored orthogonal `6 -> 3` linear material & exchange-conservation projection:
      - Row 0: Locked classical FIDE piece values (`P=1.00, N=3.05, B=3.25, R=5.00, Q=9.50, K=0.00`)
      - Row 1: Anchored Major-vs-Minor exchange conservation (`+Q, +R` vs `-N, -B, -P`)
      - Row 2: Learned piece-count / endgame phase balance
    """

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(3, 6))
        self.register_buffer("anchor0", torch.tensor([0.0430, 0.1312, 0.1398, 0.2150, 0.4085, 0.0]))
        self.register_buffer("anchor1", torch.tensor([-0.0250, -0.1050, -0.1050, 0.1850, 0.4150, 0.0]))
        self.register_buffer("no_king_mask", torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0, 0.0]))
        with torch.no_grad():
            self.weight[0].copy_(self.anchor0)
            self.weight[1].copy_(self.anchor1)
            self.weight[2].copy_(torch.tensor([0.0550, 0.1400, 0.1550, 0.1200, 0.2200, 0.0]))

    def effective_weight(self) -> torch.Tensor:
        r0 = self.anchor0 + 0.0 * self.weight[0]
        d1 = torch.clamp(self.weight[1] - self.anchor1, -0.020, 0.020)
        r1 = (self.anchor1 + d1) * self.no_king_mask
        r2_raw = torch.clamp(self.weight[2], -0.44, 0.44) * self.no_king_mask
        l1_2 = r2_raw.abs().sum().clamp(min=0.95) / 0.95
        r2 = r2_raw / l1_2
        return torch.stack([r0, r1, r2], dim=0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        z0 = ZERO_POINT
        w = self.effective_weight()
        w0 = w[:, 0:3].view(3, 3, 1, 1)
        off0 = (z0 - w[:, 0:3].sum(dim=1) * z0).view(1, 3, 1, 1)
        p0 = ste_q8(F.conv2d(x[:, 0:3], w0) + off0, quantize=quantize)

        w1 = w[:, 3:6].view(3, 3, 1, 1)
        off1 = (z0 - w[:, 3:6].sum(dim=1) * z0).view(1, 3, 1, 1)
        p1 = ste_q8(F.conv2d(x[:, 3:6], w1) + off1, quantize=quantize)

        s8 = ste_q8(p0 + p1 - z0, quantize=quantize)
        s4 = ste_q8((4.0 / 1.25) * F.avg_pool2d(s8, 2, 2) - round(2.2 * z0, 6), quantize=quantize)
        s2 = ste_q8((4.0 / 1.60) * F.avg_pool2d(s4, 2, 2) - round(1.5 * z0, 6), quantize=quantize)
        s1 = ste_q8(4.0 * F.avg_pool2d(s2, 2, 2) - round(3.0 * z0, 6), quantize=quantize)
        return s1


class SVGChessValueCNN(nn.Module):
    """BatchNorm-free 3-Layer Quantized Chess Value Network (`2,623` params, `arch_version: 2`)
    with per-triplet mean-centered `s5v` Value Head (`logit_half_range_M ≈ 6.25`, `~15 cp/LSB`).
    """

    def __init__(self, c1: int = 18, c2: int = 27, c3: int = 36):
        super().__init__()
        self.c1, self.c2, self.c3 = c1, c2, c3
        self.mix1 = SVGPointwise1x1(6, c1, init_gain=1.1, center_input=True)
        self.dwconv1 = nn.Conv2d(c1, c1, kernel_size=5, stride=1, padding=2, groups=c1, bias=True)
        self.mix2 = SVGPointwise1x1(c1, c2, init_gain=1.15)
        self.dwconv2 = nn.Conv2d(c2, c2, kernel_size=3, stride=1, padding=1, groups=c2, bias=True)
        self.mix3 = SVGPointwise1x1(c2, c3, init_gain=1.15)
        self.dwconv3 = nn.Conv2d(c3, c3, kernel_size=2, stride=1, padding=0, groups=c3, bias=True)
        self.skip_mat = SVGLinearSkip6to3()
        self.fc = nn.Linear(c3 + 3, 1, bias=True)

        # Non-parameter calibration buffers (folded directly into `value_head.color_matrices` on export!)
        self.register_buffer("feat_mu", torch.full((c3,), 0.08))
        self.register_buffer("logit_half_range_buf", torch.tensor(6.25))

        for m in (self.dwconv1, self.dwconv2, self.dwconv3):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            with torch.no_grad():
                m.bias.fill_(0.02)
        with torch.no_grad():
            self.fc.weight.zero_()
            self.fc.weight[0, c3] = 28.0      # Classical FIDE material anchor (~300 cp/pawn)
            self.fc.weight[0, c3 + 1] = 4.5   # Major-vs-Minor exchange conservation anchor
            self.fc.bias.zero_()

    def compute_logit_half_range(self) -> torch.Tensor:
        return self.logit_half_range_buf

    def extract_trunk_features(self, x: torch.Tensor, quantize: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        skip = self.skip_mat(x, quantize=quantize)
        h = self.mix1(x, quantize=quantize)
        h = ste_q8(self.dwconv1(h), quantize=quantize)
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)

        h = self.mix2(h, quantize=quantize)
        h = ste_q8(self.dwconv2(h), quantize=quantize)
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)

        h = self.mix3(h, quantize=quantize)
        h = ste_q8(self.dwconv3(h), quantize=quantize)
        return h.flatten(1), skip.flatten(1)

    @torch.no_grad()
    def calibrate_value_head(self, loader: DataLoader, min_m: float = 6.25, max_m: float = 7.75) -> None:
        """Calibrate `feat_mu` and tight non-clipping `logit_half_range_buf` across `loader`."""
        was_train = self.training
        self.eval()
        feats = []
        for i, (xb, _) in enumerate(loader):
            if i >= 64:
                break
            hf, _ = self.extract_trunk_features(xb.to(device), quantize=True)
            feats.append(hf)
        all_h = torch.cat(feats, dim=0)
        self.feat_mu.copy_(all_h.mean(dim=0))

        # Measure 99.98th-percentile running convolutional partial-sum excursion in logits
        w_conv = self.fc.weight[0, :self.c3]
        b_eff = self.fc.bias[0] + torch.dot(self.feat_mu, w_conv)
        n_cg = self.c3 // 3
        b_pg = b_eff / float(n_cg)
        run = torch.zeros(all_h.shape[0], device=all_h.device)
        peak = torch.zeros_like(run)
        for g in range(n_cg):
            wg = w_conv[3 * g : 3 * (g + 1)]
            mug = self.feat_mu[3 * g : 3 * (g + 1)]
            term = (all_h[:, 3 * g : 3 * (g + 1)] - mug) @ wg + b_pg
            peak = torch.maximum(peak, term.abs())
            run = run + term
            peak = torch.maximum(peak, run.abs())
        q_conv = float(torch.quantile(peak, 0.9998).item())
        # Ensure +/-1.35 Queens of material + positional swing fits inside [-M, +M] with zero mid-loop clipping!
        m_cal = max(min_m, min(max_m, max(q_conv / 0.36, q_conv + 4.20)))
        self.logit_half_range_buf.fill_(round(m_cal, 4))
        if was_train:
            self.train()

    def forward(self, x: torch.Tensor, quantize: bool = True, update_mu: bool = False) -> torch.Tensor:
        h_flat, skip_flat = self.extract_trunk_features(x, quantize=quantize)
        z0 = ZERO_POINT

        if update_mu and self.training:
            with torch.no_grad():
                self.feat_mu.mul_(0.96).add_(0.04 * h_flat.detach().mean(dim=0))

        if not quantize:
            feat_c = torch.cat([h_flat, skip_flat - z0], dim=1)
            return self.fc(feat_c).squeeze(-1)

        m_half = self.compute_logit_half_range()
        two_m = 2.0 * m_half
        n_cg = self.c3 // 3
        w_conv = self.fc.weight[0, :self.c3]
        b_eff = self.fc.bias[0] + torch.dot(self.feat_mu, w_conv)
        b_pg = b_eff / float(n_cg)

        acc = None
        for g in range(n_cg):
            wg = self.fc.weight[:, 3 * g : 3 * (g + 1)]
            sl_g = wg / two_m
            mu_g = self.feat_mu[3 * g : 3 * (g + 1)]
            off_g = z0 - torch.mv(sl_g, mu_g) + (b_pg / two_m)
            part = ste_q8(h_flat[:, 3 * g : 3 * (g + 1)] @ sl_g.t() + off_g.unsqueeze(0), quantize=True)
            acc = part if acc is None else ste_q8(acc + part - z0, quantize=True)

        w_sk = self.fc.weight[:, self.c3 : self.c3 + 3]
        sl_sk = w_sk / two_m
        off_sk = z0 - sl_sk.sum(dim=1) * z0
        part_sk = ste_q8(skip_flat @ sl_sk.t() + off_sk.unsqueeze(0), quantize=True)
        acc = ste_q8(acc + part_sk - z0, quantize=True)
        return ((acc - z0) * two_m).squeeze(-1)


model = SVGChessValueCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
assert total_params == 2623, f"Architecture parameter mismatch: expected 2623, got {total_params}"
print(f"SVGChessValueCNN v2 initialized | Trainable parameters: {total_params:,} (0 BatchNorm, arch-compatible!)")


In [ ]:
# ---------------------------------------------------------------------------
# 5. Stage-1 8-Bit STE QAT + Full-Hierarchy Monotonicity, Followed by
#    Stage-2 Search-In-The-Loop Blunder Mining (`DAgger`) & Fine-Tuning
# ---------------------------------------------------------------------------

def counterfactual_material_and_threat_loss(xb: torch.Tensor, logits_orig: torch.Tensor, use_qat: bool) -> torch.Tensor:
    """Full-hierarchy ceteris-paribus regularizer enforcing strict logit drops across:
      1. Bad exchanges: `QxN (-1.75)`, `QxB (-1.70)`, `QxR (-1.15)`, `RxN (-0.55)`, `RxB (-0.50)`
      2. Pure piece hangs: `-Q (-2.40)`, `-R (-1.45)`, `-B (-0.95)`, `-N (-0.92)`, `-P (-0.28)`
      3. Pawn-attacked major piece (`Q` or `R` relocated onto a square diagonally attacked by an Opp Pawn: `-1.10`)
    """
    sub_n = min(120, xb.shape[0])
    x_cf = xb[:sub_n].clone()
    req_margin = torch.zeros(sub_n, device=xb.device, dtype=torch.float32)

    with torch.no_grad():
        third = sub_n // 3
        # Part 1: Bad exchanges (QxN, QxB, QxR, RxN, RxB)
        trades = ((4, 1, 1.75), (4, 2, 1.70), (4, 3, 1.15), (3, 1, 0.55), (3, 2, 0.50))
        for i in range(third):
            ai_ch, op_ch, m_val = trades[i % len(trades)]
            ai_sqs = torch.where(x_cf[i, ai_ch].view(64) > 0.90)[0]
            op_sqs = torch.where(x_cf[i, op_ch].view(64) < 0.10)[0]
            if len(ai_sqs) > 0 and len(op_sqs) > 0:
                x_cf[i, ai_ch].view(64)[ai_sqs[0]] = ZERO_POINT
                x_cf[i, op_ch].view(64)[op_sqs[0]] = ZERO_POINT
                req_margin[i] = m_val

        # Part 2: Pure piece/pawn drops (-Q, -R, -B, -N, -P)
        drops = ((4, 2.40), (3, 1.45), (2, 0.95), (1, 0.92), (0, 0.28))
        for j in range(third, 2 * third):
            ch, m_val = drops[j % len(drops)]
            pcs = torch.where(x_cf[j, ch].view(64) > 0.90)[0]
            if len(pcs) > 0:
                x_cf[j, ch].view(64)[pcs[0]] = ZERO_POINT
                req_margin[j] = m_val

        # Part 3: Relocate an AI Queen/Rook onto an empty square attacked by an Opponent Pawn (`y+1, x+-1`)!
        for k in range(2 * third, sub_n):
            ch = 4 if (k % 2 == 0) else 3
            maj_yx = torch.where(x_cf[k, ch] > 0.90)
            op_p_yx = torch.where(x_cf[k, 0] < 0.10)
            if len(maj_yx[0]) > 0 and len(op_p_yx[0]) > 0:
                py, px = int(op_p_yx[0][0]), int(op_p_yx[1][0])
                ty = py - 1
                if 0 <= ty < 8:
                    for tx in (px - 1, px + 1):
                        if 0 <= tx < 8 and torch.all((x_cf[k, :, ty, tx] > 0.45) & (x_cf[k, :, ty, tx] < 0.55)):
                            oy, ox = int(maj_yx[0][0]), int(maj_yx[1][0])
                            x_cf[k, ch, oy, ox] = ZERO_POINT
                            x_cf[k, ch, ty, tx] = POS_VAL_U8 / 255.0
                            req_margin[k] = 1.15 if ch == 4 else 0.70
                            break

    active = req_margin > 0.0
    if not active.any():
        return xb.new_tensor(0.0)
    logits_cf = model(x_cf[active], quantize=use_qat)
    return F.relu(req_margin[active] - (logits_orig[:sub_n][active] - logits_cf)).mean()


@torch.no_grad()
def evaluate_value_net(loader: DataLoader, pairs_x: np.ndarray, pairs_y: np.ndarray, quantize: bool = True) -> Tuple[float, float, float]:
    model.eval()
    preds_list, tgts_list = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb, quantize=quantize)).cpu().numpy()
        preds_list.append(probs)
        tgts_list.append(yb.numpy())
    preds = np.concatenate(preds_list)
    tgts = np.concatenate(tgts_list)
    mae = float(np.mean(np.abs(preds - tgts)))
    corr = float(np.corrcoef(preds, tgts)[0, 1])

    p_outs = []
    px_flat = pairs_x.reshape(-1, 6, 8, 8)
    for i in range(0, len(px_flat), 512):
        pb = torch.from_numpy(px_flat[i : i + 512]).to(device)
        p_outs.append(torch.sigmoid(model(pb, quantize=quantize)).cpu().numpy())
    p_out = np.concatenate(p_outs, axis=0).reshape(-1, 2)
    diff_true = pairs_y[:, 0] - pairs_y[:, 1]
    mask_sig = np.abs(diff_true) >= 0.06
    pair_acc = float(np.mean(np.sign(p_out[mask_sig, 0] - p_out[mask_sig, 1]) == np.sign(diff_true[mask_sig]))) if mask_sig.any() else 0.0
    return mae, corr, pair_acc


# --- STAGE 1: 24 Epochs (2 Warmup + 22 Calibrated 8-Bit STE QAT) ---
STAGE1_EPOCHS = 24
optimizer = torch.optim.AdamW(model.parameters(), lr=3.0e-3, weight_decay=1.5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS, eta_min=8e-5)

history = {"train_loss": [], "val_mae": [], "val_corr": [], "val_pair_acc": []}
best_score = -1e9
best_state = None
best_metrics = (1.0, 0.0, 0.0)

for epoch in range(1, STAGE1_EPOCHS + 1):
    use_qat = (epoch >= 3)
    if epoch == 3:
        model.calibrate_value_head(train_loader, min_m=6.25, max_m=7.50)
        print(f"  [QAT Calibration] Locked Stage-1 logit_half_range_M = +-{float(model.compute_logit_half_range().item()):.3f}")

    model.train()
    run_loss, steps = 0.0, 0
    pair_iter = iter(pair_loader)

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        logits = model(xb, quantize=use_qat, update_mu=(not use_qat))
        probs = torch.sigmoid(logits)
        loss_wdl = F.binary_cross_entropy_with_logits(logits, yb) + 0.55 * F.smooth_l1_loss(probs, yb, beta=0.04)

        try:
            xpb, ypb = next(pair_iter)
        except StopIteration:
            pair_iter = iter(pair_loader)
            xpb, ypb = next(pair_iter)
        xpb, ypb = xpb.to(device), ypb.to(device)
        lp = model(xpb.view(-1, 6, 8, 8), quantize=use_qat).view(-1, 2)
        loss_pair_wdl = F.binary_cross_entropy_with_logits(lp, ypb)
        gap = ypb[:, 0] - ypb[:, 1]
        sign_t = torch.sign(gap)
        weight_pair = (gap.abs() >= 0.03).float() * (1.0 + 1.4 * (gap.abs() >= 0.22).float())
        margin = torch.clamp(gap.abs() * 3.8, max=2.0)
        loss_rank = (weight_pair * F.relu(margin - sign_t * (lp[:, 0] - lp[:, 1]))).mean()

        loss_mono = counterfactual_material_and_threat_loss(xb, logits, use_qat=use_qat)
        # Keep convolutional head weights compact so [-M, +M] stays free of mid-loop saturation
        w_conv_l1 = model.fc.weight[0, :model.c3].abs().sum()
        loss_fc_reg = F.relu(w_conv_l1 - 15.0) * 0.015

        loss = loss_wdl + 0.60 * loss_pair_wdl + 0.55 * loss_rank + 0.30 * loss_mono + loss_fc_reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.5)
        optimizer.step()

        run_loss += loss.item()
        steps += 1

    scheduler.step()
    val_mae, val_corr, val_pacc = evaluate_value_net(val_loader, xp_val_f, yp_val_f, quantize=True)
    history["train_loss"].append(run_loss / max(1, steps))
    history["val_mae"].append(val_mae)
    history["val_corr"].append(val_corr)
    history["val_pair_acc"].append(val_pacc)

    score = val_corr + 1.35 * val_pacc - 2.0 * val_mae
    marker = ""
    if use_qat and score >= best_score:
        best_score = score
        best_state = deepcopy(model.state_dict())
        best_metrics = (val_mae, val_corr, val_pacc)
        marker = " * [Stage-1 best]"

    mode_s = "8-bit QAT" if use_qat else "Warmup   "
    print(
        f"[Stage 1] Epoch {epoch:02d}/{STAGE1_EPOCHS:02d} ({mode_s}) | "
        f"Loss: {history['train_loss'][-1]:.4f} | Val MAE: {val_mae:.4f} | "
        f"SF r: {val_corr:.4f} | Pair Acc: {val_pacc * 100:5.2f}%{marker}"
    )

model.load_state_dict(best_state)
print(f"\nStage 1 complete -> Val MAE: {best_metrics[0]:.4f} | SF r: {best_metrics[1]:.4f} | Pair Acc: {best_metrics[2]*100:.2f}%\n")


# ---------------------------------------------------------------------------
# STAGE 2: Search-In-The-Loop Blunder Mining (`DAgger`) + 12 QAT Fine-Tune Epochs!
# ---------------------------------------------------------------------------
@torch.no_grad()
def _gpu_eval_u8_for_search(leaves_u8: np.ndarray) -> np.ndarray:
    model.eval()
    m_half = float(model.compute_logit_half_range().item())
    two_m = 2.0 * m_half
    grid = np.arange(256, dtype=np.float64) / 255.0
    sig_p = 1.0 / (1.0 + np.exp(-np.clip((grid - ZERO_POINT) * two_m, -20.0, 20.0)))
    lut_u8 = np.clip(np.round(3.0 + 250.0 * sig_p), 3.0, 253.0).astype(np.int32)

    out = np.empty(len(leaves_u8), dtype=np.int32)
    for i in range(0, len(leaves_u8), 2048):
        xb = torch.from_numpy(leaves_u8[i : i + 2048].astype(np.float32) / 255.0).to(device)
        lg = model(xb, quantize=True).cpu().numpy()
        acc_idx = np.clip(np.round((lg / two_m + ZERO_POINT) * 255.0).astype(np.int32), 0, 255)
        out[i : i + 2048] = lut_u8[acc_idx]
    return out


def mine_search_blunders_dagger(sf_path: str, n_roots: int = 2800) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Run the live quantized `2-Ply + Ply-3 Quiescence` engine on `n_roots` opening & tactical boards,
    query Stockfish on `(engine_move vs sf_best_move)`, and harvest every >= 55 cp blunder into exact minimax leaf pairs!
    """
    t_m = time.time()
    rng_m = random.Random(SEED + 777)
    root_boards: List[chess.Board] = []
    seen_rf = set()

    # 50% opening & early-middlegame boards across all 48 book stems + 1-ply deviations
    for stem in OPENING_BOOK_UCI:
        b = chess.Board()
        for uci in stem:
            if not b.is_game_over():
                fk = b.board_fen() + (" w" if b.turn == chess.WHITE else " b")
                if fk not in seen_rf:
                    seen_rf.add(fk)
                    root_boards.append(b.copy(stack=False))
                for alt in rng_m.sample(list(b.legal_moves), min(3, b.legal_moves.count())):
                    ba = b.copy(stack=False); ba.push(alt)
                    afk = ba.board_fen() + (" w" if ba.turn == chess.WHITE else " b")
                    if not ba.is_game_over() and afk not in seen_rf and len(root_boards) < n_roots // 2:
                        seen_rf.add(afk)
                        root_boards.append(ba)
            mv = chess.Move.from_uci(uci)
            if mv in b.legal_moves:
                b.push(mv)

    # Fill remaining 50% from unseen middlegame/tactical positions
    cand_pool = [f for f, aw, r in SINGLETONS[: n_train] if (f.split(" ")[1] == "w") == aw and r in ("sf_play", "opening", "bad_pos", "hang_or_q3")]
    rng_m.shuffle(cand_pool)
    for fen in cand_pool:
        if len(root_boards) >= n_roots:
            break
        bd = chess.Board(fen)
        if not bd.is_game_over() and bd.legal_moves.count() >= 4:
            fk = bd.board_fen() + (" w" if bd.turn == chess.WHITE else " b")
            if fk not in seen_rf:
                seen_rf.add(fk)
                root_boards.append(bd)

    print(f"[Stage 2 DAgger] Running quantized 2-Ply + Ply-3 Quiescence search across {len(root_boards):,} root boards...")
    search_outs = []
    for i in range(0, len(root_boards), 160):
        search_outs.extend(select_move_2ply_q3_svg_batch(root_boards[i : i + 160], _gpu_eval_u8_for_search))
    print(f"  2-Ply + Q3 GPU search finished in {time.time() - t_m:.1f}s; auditing moves against Stockfish...")

    num_workers = max(2, min(6, (os.cpu_count() or 2)))
    chunk_sz = math.ceil(len(root_boards) / num_workers)

    def _audit_worker(s_i: int, e_i: int) -> Tuple[List[PairInstance], List[ Tuple[str, bool] ], int]:
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        loc_pairs: List[PairInstance] = []
        loc_sings: List[Tuple[str, bool]] = []
        n_blund = 0
        try:
            eng.configure({"Threads": 1, "Hash": 24})
            lim_chk = chess.engine.Limit(nodes=1600)
            for idx in range(s_i, e_i):
                bd = root_boards[idx]
                ai_col = bd.turn
                ai_w = (ai_col == chess.WHITE)
                mv_svg, _q_svg, uci_scores, uci_leaves = search_outs[idx]

                infos = eng.analyse(bd, lim_chk, multipv=min(3, len(uci_leaves)))
                if not infos or "pv" not in infos[0] or not infos[0]["pv"]:
                    continue
                mv_sf = infos[0]["pv"][0]
                if mv_sf.uci() not in uci_leaves:
                    continue

                sc_best = infos[0]["score"].pov(ai_col)
                cp_best = 2500.0 if (sc_best.mate() is not None and sc_best.mate() > 0) else (-2500.0 if sc_best.mate() is not None else float(sc_best.score() or 0.0))

                # Evaluate the engine's actual chosen move `mv_svg`
                b_svg = bd.copy(stack=False)
                b_svg.push(mv_svg)
                if b_svg.is_checkmate():
                    cp_svg = 2500.0
                elif b_svg.is_stalemate() or b_svg.is_insufficient_material():
                    cp_svg = 0.0
                else:
                    inf_svg = eng.analyse(b_svg, lim_chk)
                    ps = inf_svg["score"].pov(ai_col)
                    cp_svg = 2500.0 if (ps.mate() is not None and ps.mate() > 0) else (-2500.0 if ps.mate() is not None else float(ps.score() or 0.0))

                if mv_svg != mv_sf and (cp_best - cp_svg) >= 55.0:
                    n_blund += 1
                    leaf_good_b = uci_leaves[mv_sf.uci()]
                    leaf_bad_b = uci_leaves[mv_svg.uci()]
                    lg_inst = (leaf_good_b.fen(), ai_w)
                    lb_inst = (leaf_bad_b.fen(), ai_w)
                    loc_pairs.append((lg_inst, lb_inst))
                    loc_sings.append(lg_inst)
                    loc_sings.append(lb_inst)

                    # Also check runner-up high-Q move if it was ALSO a blunder!
                    top_ucis = sorted(uci_scores.keys(), key=lambda u: uci_scores[u], reverse=True)
                    for alt_u in top_ucis[:3]:
                        if alt_u not in (mv_svg.uci(), mv_sf.uci()):
                            b_alt = bd.copy(stack=False); b_alt.push(chess.Move.from_uci(alt_u))
                            if not b_alt.is_game_over():
                                ps2 = eng.analyse(b_alt, chess.engine.Limit(nodes=900))["score"].pov(ai_col)
                                cp2 = 2500.0 if (ps2.mate() and ps2.mate() > 0) else (-2500.0 if ps2.mate() else float(ps2.score() or 0.0))
                                if (cp_best - cp2) >= 90.0:
                                    la_inst = (uci_leaves[alt_u].fen(), ai_w)
                                    loc_pairs.append((lg_inst, la_inst))
                                    loc_sings.append(la_inst)
                                    break
        finally:
            eng.quit()
        return loc_pairs, loc_sings, n_blund

    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futs = [pool.submit(_audit_worker, w * chunk_sz, min(len(root_boards), (w + 1) * chunk_sz)) for w in range(num_workers) if w * chunk_sz < len(root_boards)]
        aud_res = [f.result() for f in futs]

    mined_pairs: List[PairInstance] = []
    mined_sings: List[Tuple[str, bool]] = []
    total_blunders = 0
    for lp, ls, nb in aud_res:
        mined_pairs.extend(lp)
        mined_sings.extend(ls)
        total_blunders += nb

    print(
        f"[Stage 2 DAgger] Caught {total_blunders:,}/{len(root_boards):,} search blunders (>=55cp) -> "
        f"harvested {len(mined_pairs):,} hard-negative minimax leaf pairs!"
    )
    if not mined_pairs:
        return np.zeros((0, 2, 6, 8, 8), dtype=np.float32), np.zeros((0, 2), dtype=np.float32), np.zeros((0, 6, 8, 8), dtype=np.float32), np.zeros((0,), dtype=np.float32)

    flat_p = [inst for pr in mined_pairs for inst in pr]
    comb_m = mined_sings + flat_p
    xm_u8, ym_wdl = label_instances_with_stockfish(comb_m, sf_path, nodes_limit=2800)
    ns_m = len(mined_sings)
    xs_m = xm_u8[:ns_m].astype(np.float32) / 255.0
    ys_m = ym_wdl[:ns_m]
    xp_m = xm_u8[ns_m:].reshape(len(mined_pairs), 2, 6, 8, 8).astype(np.float32) / 255.0
    yp_m = ym_wdl[ns_m:].reshape(len(mined_pairs), 2)
    return xp_m, yp_m, xs_m, ys_m


xp_dagger, yp_dagger, xs_dagger, ys_dagger = mine_search_blunders_dagger(STOCKFISH_PATH, n_roots=2800)

if len(xp_dagger) >= 16:
    n_dg_tr = max(8, int(0.88 * len(xp_dagger)))
    xp_dg_tr, yp_dg_tr = xp_dagger[:n_dg_tr], yp_dagger[:n_dg_tr]
    xp_dg_va, yp_dg_va = (xp_dagger[n_dg_tr:], yp_dagger[n_dg_tr:]) if len(xp_dagger) - n_dg_tr >= 4 else (xp_dagger[:16], yp_dagger[:16])

    # Blend Stage-1 data with 3x oversampled DAgger mined search-blunder leaves!
    x_s2_tr = np.concatenate([x_train_f, np.repeat(xs_dagger, 3, axis=0)], axis=0)
    y_s2_tr = np.concatenate([y_train_f, np.repeat(ys_dagger, 3, axis=0)], axis=0)
    s2_train_loader = DataLoader(TensorDataset(torch.from_numpy(x_s2_tr), torch.from_numpy(y_s2_tr)), batch_size=BATCH_SIZE, shuffle=True)
    dg_pair_loader = DataLoader(TensorDataset(torch.from_numpy(xp_dg_tr), torch.from_numpy(yp_dg_tr)), batch_size=min(96, len(xp_dg_tr)), shuffle=True)

    STAGE2_EPOCHS = 12
    model.calibrate_value_head(s2_train_loader, min_m=6.25, max_m=7.50)
    print(f"\n[Stage 2 QAT] Locked final logit_half_range_M = +-{float(model.compute_logit_half_range().item()):.3f}")

    opt_s2 = torch.optim.AdamW(model.parameters(), lr=6.5e-4, weight_decay=1.5e-4)
    sch_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s2, T_max=STAGE2_EPOCHS, eta_min=3e-5)

    best_s2_score = -1e9
    best_s2_state = deepcopy(model.state_dict())

    for ep2 in range(1, STAGE2_EPOCHS + 1):
        model.train()
        r_loss, st2 = 0.0, 0
        p1_iter = iter(pair_loader)
        dg_iter = iter(dg_pair_loader)

        for xb, yb in s2_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt_s2.zero_grad()

            logits = model(xb, quantize=True, update_mu=False)
            probs = torch.sigmoid(logits)
            loss_wdl = F.binary_cross_entropy_with_logits(logits, yb) + 0.55 * F.smooth_l1_loss(probs, yb, beta=0.04)

            try:
                xpb, ypb = next(p1_iter)
            except StopIteration:
                p1_iter = iter(pair_loader)
                xpb, ypb = next(p1_iter)
            xpb, ypb = xpb.to(device), ypb.to(device)
            lp1 = model(xpb.view(-1, 6, 8, 8), quantize=True).view(-1, 2)
            gap1 = ypb[:, 0] - ypb[:, 1]
            loss_p1 = F.binary_cross_entropy_with_logits(lp1, ypb) + (
                (gap1.abs() >= 0.03).float() * F.relu(torch.clamp(gap1.abs() * 3.8, max=2.0) - torch.sign(gap1) * (lp1[:, 0] - lp1[:, 1]))
            ).mean()

            try:
                xdgb, ydgb = next(dg_iter)
            except StopIteration:
                dg_iter = iter(dg_pair_loader)
                xdgb, ydgb = next(dg_iter)
            xdgb, ydgb = xdgb.to(device), ydgb.to(device)
            lpd = model(xdgb.view(-1, 6, 8, 8), quantize=True).view(-1, 2)
            gapd = ydgb[:, 0] - ydgb[:, 1]
            loss_dg = F.binary_cross_entropy_with_logits(lpd, ydgb) + 1.35 * (
                (gapd.abs() >= 0.03).float() * F.relu(torch.clamp(gapd.abs() * 4.2, max=2.2) - torch.sign(gapd) * (lpd[:, 0] - lpd[:, 1]))
            ).mean()

            loss_mono = counterfactual_material_and_threat_loss(xb, logits, use_qat=True)
            w_conv_l1 = model.fc.weight[0, :model.c3].abs().sum()
            loss_fc_reg = F.relu(w_conv_l1 - 15.0) * 0.015

            loss = loss_wdl + 0.45 * loss_p1 + 0.65 * loss_dg + 0.30 * loss_mono + loss_fc_reg
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt_s2.step()

            r_loss += loss.item()
            st2 += 1

        sch_s2.step()
        val_mae, val_corr, val_pacc = evaluate_value_net(val_loader, xp_val_f, yp_val_f, quantize=True)
        _, _, dg_pacc = evaluate_value_net(val_loader, xp_dg_va, yp_dg_va, quantize=True)
        history["train_loss"].append(r_loss / max(1, st2))
        history["val_mae"].append(val_mae)
        history["val_corr"].append(val_corr)
        history["val_pair_acc"].append(val_pacc)

        sc2_tot = val_corr + 1.25 * val_pacc + 0.45 * dg_pacc - 2.0 * val_mae
        mk = ""
        if sc2_tot >= best_s2_score:
            best_s2_score = sc2_tot
            best_s2_state = deepcopy(model.state_dict())
            best_metrics = (val_mae, val_corr, val_pacc)
            mk = " * [Stage-2 best]"
        print(
            f"[Stage 2] Epoch {ep2:02d}/{STAGE2_EPOCHS:02d} (DAgger QAT) | "
            f"Loss: {history['train_loss'][-1]:.4f} | Val MAE: {val_mae:.4f} | "
            f"SF r: {val_corr:.4f} | Pair Acc: {val_pacc*100:5.2f}% | Mined Blunder Fix: {dg_pacc*100:5.2f}%{mk}"
        )

    model.load_state_dict(best_s2_state)

print(
    f"\nLoaded Final Best QAT Checkpoint -> Val MAE: {best_metrics[0]:.4f} | "
    f"Stockfish r: {best_metrics[1]:.4f} | 2-Ply+Q3 Contrastive Ranking: {best_metrics[2] * 100:.2f}%"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.4))
ep_x = range(1, len(history["train_loss"]) + 1)
ax1.plot(ep_x, history["train_loss"], label="Train Loss")
ax1.plot(ep_x, history["val_mae"], label="Val WDL MAE")
ax1.axvline(STAGE1_EPOCHS + 0.5, color="#f43f5e", linestyle="--", alpha=0.7, label="DAgger Stage 2")
ax1.set_xlabel("Epoch"); ax1.set_title("Stage 1 + DAgger Stage 2 Loss & Calibration"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_x, [c * 100 for c in history["val_corr"]], label="Stockfish Pearson r (%)")
ax2.plot(ep_x, [p * 100 for p in history["val_pair_acc"]], label="2-Ply+Q3 Pair Rank Acc (%)")
ax2.axvline(STAGE1_EPOCHS + 0.5, color="#f43f5e", linestyle="--", alpha=0.7, label="DAgger Stage 2")
ax2.set_xlabel("Epoch"); ax2.set_title("8-Bit QAT Correlation & Anti-Blunder Ranking"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 6. Compile Trained Model to Drop-In `nojs-svg-chess-value-cnn-v1` Spec
#    + Bit-Exact 8-Bit NumPy SVG Simulator & 18-Position Anti-Blunder Gate
# ---------------------------------------------------------------------------

def flip_kernel_2d(k2d: np.ndarray) -> List[float]:
    return [round(float(v), 6) for v in k2d[::-1, ::-1].reshape(-1)]


def export_depthwise_conv(conv: nn.Conv2d) -> dict:
    w = conv.weight.detach().cpu().numpy()
    b = conv.bias.detach().cpu().numpy()
    kh, kw = int(w.shape[2]), int(w.shape[3])
    tx = (kw - 1) // 2 if conv.padding[1] > 0 else 0
    ty = (kh - 1) // 2 if conv.padding[0] > 0 else 0
    filters = []
    for c in range(w.shape[0]):
        filters.append({
            "channel": int(c),
            "order": [kw, kh],
            "targetX": int(tx),
            "targetY": int(ty),
            "divisor": 1.0,
            "bias": round(float(b[c]), 6),
            "kernelMatrix": flip_kernel_2d(w[c, 0]),
            "padding": int(conv.padding[0]),
        })
    return {
        "num_channels": int(w.shape[0]),
        "svg_order": [kw, kh],
        "targetX": int(tx),
        "targetY": int(ty),
        "padding": int(conv.padding[0]),
        "filters": filters,
    }


def export_pointwise(pw: SVGPointwise1x1) -> dict:
    w = pw.weight.detach().cpu().numpy()
    b = pw.bias.detach().cpu().numpy()
    s_ch = pw.compute_channel_scales().cpu().numpy()
    z0 = ZERO_POINT
    g_in = pw.in_channels // 3
    g_out = pw.out_channels // 3
    triplets = []
    for o in range(g_out):
        s_vec = [float(s_ch[3 * o + r]) for r in range(3)]
        b_slice = b[3 * o : 3 * (o + 1)]
        color_matrices = []
        for g in range(g_in):
            blk = w[3 * o : 3 * (o + 1), 3 * g : 3 * (g + 1)] / np.array(s_vec, dtype=np.float64)[:, None]
            mat20 = [
                round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, z0,
                round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, z0,
                round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, z0,
                0.0, 0.0, 0.0, 0.0, 1.0,
            ]
            color_matrices.append(mat20)
        triplets.append({
            "out_triplet": int(o),
            "scales": [round(s, 6) for s in s_vec],
            "rescale_slopes": [round(s, 6) for s in s_vec],
            "rescale_intercepts": [round(float(b_slice[r] - z0 * s_vec[r]), 6) for r in range(3)],
            "color_matrices": color_matrices,
        })
    return {"in_channels": pw.in_channels, "out_channels": pw.out_channels, "zero_point": z0, "triplets": triplets}


def export_linear_skip(skip: SVGLinearSkip6to3) -> dict:
    w = skip.effective_weight().detach().cpu().numpy()
    z0 = ZERO_POINT
    mats = []
    for g in range(2):
        blk = w[:, 3 * g : 3 * (g + 1)]
        offs = [round(float(z0 - blk[r].sum() * z0), 6) for r in range(3)]
        mats.append([
            round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, offs[0],
            round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, offs[1],
            round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, offs[2],
            0.0, 0.0, 0.0, 0.0, 1.0,
        ])
    return {
        "zero_point": z0,
        "weights_3x6": [[round(float(x), 6) for x in row] for row in w],
        "color_matrices": mats,
        "sum_stages": [
            {"order": [2, 2], "divisor": 1.25, "bias": round(-2.2 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.60, "bias": round(-1.5 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.00, "bias": round(-3.0 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
        ],
    }


def export_value_head(model: SVGChessValueCNN) -> dict:
    """Export per-triplet mean-centered `value_head` (`0` changes to `svg_ai.py` `<filter id='ai-s5v-*'>`!)."""
    w_fc = model.fc.weight.detach().cpu().numpy()[0]
    b_fc = float(model.fc.bias.detach().cpu().item())
    mu_np = model.feat_mu.detach().cpu().numpy()
    z0 = ZERO_POINT
    m_half = float(model.compute_logit_half_range().cpu().item())
    two_m = 2.0 * m_half
    n_cg = model.c3 // 3
    g_in = len(w_fc) // 3
    b_eff = b_fc + float(np.dot(mu_np, w_fc[: model.c3]))
    b_pg = b_eff / float(n_cg)

    color_matrices = []
    for g in range(g_in):
        sl = w_fc[3 * g : 3 * (g + 1)] / two_m
        if g < n_cg:
            mu_g = mu_np[3 * g : 3 * (g + 1)]
            off = z0 - float(np.dot(mu_g, sl)) + (b_pg / two_m)
        else:
            off = z0 - float(sl.sum()) * z0
        row = [round(float(sl[0]), 6), round(float(sl[1]), 6), round(float(sl[2]), 6), 0.0, round(float(off), 6)]
        color_matrices.append(row + row + row + [0.0, 0.0, 0.0, 0.0, 1.0])

    grid = np.arange(256, dtype=np.float64) / 255.0
    logits_grid = (grid - z0) * two_m
    sig_probs = 1.0 / (1.0 + np.exp(-np.clip(logits_grid, -20.0, 20.0)))
    v_table_u8 = np.clip(np.round(3.0 + 250.0 * sig_probs), 3.0, 253.0) / 255.0

    return {
        "in_features": int(len(w_fc)),
        "zero_point": z0,
        "u8_min": 3,
        "u8_max": 253,
        "logit_half_range_M": round(m_half, 6),
        "color_matrices": color_matrices,
        "sigmoid_table_values": [round(float(v), 6) for v in v_table_u8],
    }


def build_chess_export_spec(model: SVGChessValueCNN) -> dict:
    return {
        "format": "nojs-svg-chess-value-cnn-v1",
        "arch_version": 2,
        "zero_point": ZERO_POINT,
        "zero_point_u8": ZERO_POINT_U8,
        "pos_val_u8": POS_VAL_U8,
        "neg_val_u8": NEG_VAL_U8,
        "input_shape": [6, 8, 8],
        "channel_order": ["PAWN", "KNIGHT", "BISHOP", "ROOK", "QUEEN", "KING"],
        "layers": {
            "mix1": export_pointwise(model.mix1),
            "dwconv1": export_depthwise_conv(model.dwconv1),
            "pool1": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix2": export_pointwise(model.mix2),
            "dwconv2": export_depthwise_conv(model.dwconv2),
            "pool2": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix3": export_pointwise(model.mix3),
            "dwconv3": export_depthwise_conv(model.dwconv3),
            "skip_mat": export_linear_skip(model.skip_mat),
            "value_head": export_value_head(model),
        },
    }


def q8_np(x: np.ndarray) -> np.ndarray:
    return np.round(np.clip(x, 0.0, 1.0) * 255.0) / 255.0


def svg_lut_eval_np(x: np.ndarray, table_vals: Sequence[float]) -> np.ndarray:
    tbl = np.asarray(table_vals, dtype=np.float64)
    n = len(tbl) - 1
    pos = np.clip(x, 0.0, 1.0) * n
    idx = np.minimum(np.floor(pos).astype(np.int64), n - 1)
    frac = pos - idx
    return q8_np(tbl[idx] * (1.0 - frac) + tbl[idx + 1] * frac)


def svg_fe_convolve_batch_np(buf: np.ndarray, order: Sequence[int], tx: int, ty: int,
                             kernel_flat: Sequence[float], divisor: float, bias: float, stride: int = 1) -> np.ndarray:
    ow, oh = int(order[0]), int(order[1])
    k = np.asarray(kernel_flat, dtype=np.float64).reshape(oh, ow)
    bsz, h, w = buf.shape
    padded = np.pad(buf, ((0, 0), (ty, oh - 1 - ty), (tx, ow - 1 - tx)), mode="constant", constant_values=0.0)
    acc = np.zeros((bsz, h, w), dtype=np.float64)
    for i in range(oh):
        for j in range(ow):
            kv = k[oh - 1 - i, ow - 1 - j]
            if kv != 0.0:
                acc += kv * padded[:, i : i + h, j : j + w]
    res = q8_np(acc / float(divisor) + float(bias))
    if stride > 1:
        return res[:, 0::stride, 0::stride]
    if oh == 2 and ow == 2 and tx == 0 and ty == 0 and h == 2 and w == 2:
        return res[:, 0:1, 0:1]
    return res


def svg_pointwise_batch_np(ch_bufs: np.ndarray, pw_spec: dict) -> np.ndarray:
    bsz, _, h, w = ch_bufs.shape
    z0 = float(pw_spec["zero_point"])
    out = np.zeros((bsz, pw_spec["out_channels"], h, w), dtype=np.float64)
    for trip in pw_spec["triplets"]:
        o = trip["out_triplet"]
        acc = None
        for g, mat20 in enumerate(trip["color_matrices"]):
            m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[:3, :]
            rgb_in = ch_bufs[:, 3 * g : 3 * (g + 1)]
            part = q8_np(np.einsum("rc,bchw->brhw", m[:, :3], rgb_in) + m[:, 4].reshape(1, 3, 1, 1))
            acc = part if acc is None else q8_np(acc + part - z0)
        slopes = np.asarray(trip["rescale_slopes"], dtype=np.float64).reshape(1, 3, 1, 1)
        intercepts = np.asarray(trip["rescale_intercepts"], dtype=np.float64).reshape(1, 3, 1, 1)
        out[:, 3 * o : 3 * (o + 1)] = q8_np(acc * slopes + intercepts)
    return out


def simulate_svg_value_batch(x_float_6x8x8: np.ndarray, spec: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bit-exact mirror of `chess/svg_ai.py::evaluate_svg_cnn_batch`."""
    ly = spec["layers"]
    x0 = q8_np(x_float_6x8x8)

    sk_spec = ly["skip_mat"]
    z0 = float(sk_spec["zero_point"])
    m0 = np.asarray(sk_spec["color_matrices"][0], dtype=np.float64).reshape(4, 5)[:3, :]
    m1 = np.asarray(sk_spec["color_matrices"][1], dtype=np.float64).reshape(4, 5)[:3, :]
    p0 = q8_np(np.einsum("rc,bchw->brhw", m0[:, :3], x0[:, 0:3]) + m0[:, 4].reshape(1, 3, 1, 1))
    p1 = q8_np(np.einsum("rc,bchw->brhw", m1[:, :3], x0[:, 3:6]) + m1[:, 4].reshape(1, 3, 1, 1))
    sk = q8_np(p0 + p1 - z0)
    for stg in sk_spec["sum_stages"]:
        stride = 2 if sk.shape[-1] > 2 else 1
        sk = np.stack([
            svg_fe_convolve_batch_np(sk[:, c], stg["order"], 0, 0, stg["kernelMatrix"], stg["divisor"], stg["bias"], stride=stride)
            for c in range(3)
        ], axis=1)

    h = svg_pointwise_batch_np(x0, ly["mix1"])
    h = np.stack([svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"]) for f in ly["dwconv1"]["filters"]], axis=1)
    p1_s = ly["pool1"]
    h = np.stack([svg_fe_convolve_batch_np(h[:, c], p1_s["order"], 0, 0, p1_s["kernelMatrix"], p1_s["divisor"], 0.0, stride=2) for c in range(h.shape[1])], axis=1)

    h = svg_pointwise_batch_np(h, ly["mix2"])
    h = np.stack([svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"]) for f in ly["dwconv2"]["filters"]], axis=1)
    p2_s = ly["pool2"]
    h = np.stack([svg_fe_convolve_batch_np(h[:, c], p2_s["order"], 0, 0, p2_s["kernelMatrix"], p2_s["divisor"], 0.0, stride=2) for c in range(h.shape[1])], axis=1)

    h = svg_pointwise_batch_np(h, ly["mix3"])
    h = np.stack([svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"]) for f in ly["dwconv3"]["filters"]], axis=1)

    feat = np.concatenate([h[:, :, 0, 0], sk[:, :, 0, 0]], axis=1)
    vh = ly["value_head"]
    z0_vh = float(vh["zero_point"])
    acc = None
    for g, mat20 in enumerate(vh["color_matrices"]):
        m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[0, :]
        part = q8_np(feat[:, 3 * g : 3 * (g + 1)] @ m[:3] + m[4])
        acc = part if acc is None else q8_np(acc + part - z0_vh)

    two_m = 2.0 * float(vh["logit_half_range_M"])
    decoded_logits = (acc - z0_vh) * two_m
    v_svg = svg_lut_eval_np(acc, vh["sigmoid_table_values"])
    v_u8 = np.round(v_svg * 255.0).astype(np.int32)
    probs = (v_svg * 255.0 - 3.0) / 250.0
    return decoded_logits, probs, v_u8


export_spec = build_chess_export_spec(model)
sig_arr = np.round(np.array(export_spec["layers"]["value_head"]["sigmoid_table_values"]) * 255.0)
active_lut_bins = int(np.sum((sig_arr > 3) & (sig_arr < 253)))
print("Compiled Drop-In SVG `<filter>` Chess Value CNN v2 Spec!")
print(f"  Logit half-range M      = +-{export_spec['layers']['value_head']['logit_half_range_M']} (Active LUT bins: {active_lut_bins}/256)")
print(f"  skip_mat row 0 (FIDE)   = {export_spec['layers']['skip_mat']['weights_3x6'][0]}")
print(f"  skip_mat row 1 (MajMin) = {export_spec['layers']['skip_mat']['weights_3x6'][1]}")
assert export_spec["layers"]["value_head"]["logit_half_range_M"] <= 8.5
assert active_lut_bins >= 200, f"Expected >= 200 active LUT bins, got {active_lut_bins}"


In [ ]:
# ---------------------------------------------------------------------------
# 7. Bit-Exact 8-Bit SVG Parity, Piece-Hierarchy Check, 18-Position Tactical/Opening
#    Suite & `weights.json` Export (Compatible with `chess/test_chess.py`)
# ---------------------------------------------------------------------------

@torch.no_grad()
def get_pytorch_qat_logits(x_np: np.ndarray) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(x_np), 512):
        xb = torch.from_numpy(x_np[i : i + 512]).to(device)
        outs.append(model(xb, quantize=True).cpu().numpy())
    return np.concatenate(outs, axis=0)


qat_logits_test = get_pytorch_qat_logits(x_test_f)
qat_probs_test = 1.0 / (1.0 + np.exp(-np.clip(qat_logits_test, -20.0, 20.0)))

svg_logits_chunks, svg_probs_chunks, svg_u8_chunks = [], [], []
for i in range(0, len(x_test_f), 1024):
    lg_c, pr_c, u8_c = simulate_svg_value_batch(x_test_f[i : i + 1024], export_spec)
    svg_logits_chunks.append(lg_c)
    svg_probs_chunks.append(pr_c)
    svg_u8_chunks.append(u8_c)
svg_logits_test = np.concatenate(svg_logits_chunks)
svg_probs_test = np.concatenate(svg_probs_chunks)
svg_u8_test = np.concatenate(svg_u8_chunks)

logit_abs_diff = np.abs(qat_logits_test - svg_logits_test)
lsb_step = (2.0 * float(export_spec["layers"]["value_head"]["logit_half_range_M"])) / 255.0
lsb_diff = np.round(logit_abs_diff / max(lsb_step, 1e-9))

test_mae_qat = float(np.mean(np.abs(qat_probs_test - y_test_f)))
test_corr_qat = float(np.corrcoef(qat_probs_test, y_test_f)[0, 1])
test_mae_svg = float(np.mean(np.abs(svg_probs_test - y_test_f)))
test_corr_svg = float(np.corrcoef(svg_probs_test, y_test_f)[0, 1])
qat_svg_mae_gap = abs(test_mae_qat - test_mae_svg)
qat_svg_corr_gap = abs(test_corr_qat - test_corr_svg)

print("=====================================================================")
print(f"  Held-Out Test Set Size:                 {len(y_test_f):,} unseen Stockfish instances")
print(f"  PyTorch 8-Bit QAT WDL MAE / Corr r:     {test_mae_qat:.4f} / {test_corr_qat:.4f}")
print(f"  Exact 8-Bit SVG WDL MAE / Corr r:       {test_mae_svg:.4f} / {test_corr_svg:.4f}  (gap: {qat_svg_mae_gap:.4f} / {qat_svg_corr_gap:.4f})")
print(f"  FP32-QAT vs FP64-SVG Byte Diff (LSBs):  Median {np.median(lsb_diff):.0f} LSB | 99th pct {np.percentile(lsb_diff, 99):.0f} LSB | Max {lsb_diff.max():.0f} LSB")
print("=====================================================================")
assert test_corr_svg >= 0.88, f"Expected >= 0.88 Stockfish Pearson r, got {test_corr_svg:.4f}"
assert qat_svg_mae_gap < 0.008 and qat_svg_corr_gap < 0.008
assert np.median(lsb_diff) <= 1


def _exact_svg_eval_u8(leaves_u8: np.ndarray) -> np.ndarray:
    outs = []
    for i in range(0, len(leaves_u8), 512):
        _, _, u8 = simulate_svg_value_batch(leaves_u8[i : i + 512].astype(np.float32) / 255.0, export_spec)
        outs.append(u8)
    return np.concatenate(outs, axis=0)


# Ceteris-Paribus Piece-Drop LSB Audit across 4 balanced middlegame boards
probe_fens = [
    "r1bqkb1r/ppp2ppp/2np1n2/4p3/2B1P3/2NP1N2/PPP2PPP/R1BQK2R b KQkq - 0 5",
    "rnbq1rk1/ppp1ppbp/3p1np1/8/2PPP3/2N2N2/PP2BPPP/R1BQK2R b KQ - 2 6",
    "r2qkb1r/pp2pppp/2n2n2/2pp1b2/3P1B2/2P1PN2/PP1N1PPP/R2QKB1R b KQkq - 2 6",
    "r1bq1rk1/pppp1ppp/2n2n2/2b1p3/2B1P3/2NP1N2/PPP2PPP/R1BQ1RK1 b - - 6 6",
]
probe_u8 = np.stack([encode_board_uint8(chess.Board(f)) for f in probe_fens], axis=0)
lg_base, _, u8_base = simulate_svg_value_batch(probe_u8.astype(np.float32) / 255.0, export_spec)
print("\nCeteris-Paribus 8-Bit SVG Material Sensitivity:")
mean_drops = {}
for ch, nm in [(0, "Pawn"), (1, "Knight"), (2, "Bishop"), (3, "Rook"), (4, "Queen")]:
    b_dr = probe_u8.copy()
    for i in range(len(b_dr)):
        ys, xs = np.where(b_dr[i, ch] == POS_VAL_U8)
        b_dr[i, ch, ys[0], xs[0]] = ZERO_POINT_U8
    lg_d, _, u8_d = simulate_svg_value_batch(b_dr.astype(np.float32) / 255.0, export_spec)
    mean_drops[nm] = float((lg_d - lg_base).mean())
    lsb_d = int(round(mean_drops[nm] / lsb_step))
    print(f"  Drop 1 AI {nm:6s}: mean d_logit = {mean_drops[nm]:+.3f} ({lsb_d:+3d} LSB) | mean out u8 delta = {(u8_d - u8_base).mean():+.1f}")

assert mean_drops["Queen"] < mean_drops["Rook"] < min(mean_drops["Bishop"], mean_drops["Knight"]) < mean_drops["Pawn"] < -0.12

# 18-Position Exact `2-Ply + Ply-3 Quiescence` Tactical, Piece-Conservation & Opening Suite (mirrors `chess/test_chess.py` + new checks!)
SUITE_2PLY_Q3 = [
    ("Capture Hanging Queen (Bxg4)", "r1bqkbnr/ppp2ppp/2np4/4p3/4P1Q1/2N2N2/PPPP1PPP/R1B1KB1R b KQkq - 1 4", "exact", {"c8g4"}),
    ("Scholar's Mate Defense (3...Nf6?? Q==2)", "r1bqkbnr/pppp1ppp/2n5/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR b KQkq - 3 3", "exact", {"d7d5", "g7g6", "d8e7", "d8f6", "g8h6"}),
    ("Save Attacked Qh4 (Refuse Qxf2??)", "rnb1kbnr/pppp1ppp/8/4p3/4P2q/2N2N2/PPPP1PPP/R1BQKB1R b KQkq - 3 3", "exact", {"h4f6", "h4f4", "h4e7", "h4d8", "h4h5", "h4h6", "h4g4"}),
    ("Defend f7 vs Qh4+Ng5", "r1bq1rk1/pppp1ppp/2n5/2b1p1N1/4P2Q/3P4/PPP2PPP/RNB2RK1 b - - 4 6", "exact", {"f8e8", "h7h6", "h7h5"}),
    ("Back-Rank Mate in 1 (Rd8#)", "6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1", "exact", {"d1d8"}),
    ("Win Undefended Rook (Bxh1)", "4k3/p7/8/3b4/8/8/8/4K2R b - - 0 1", "exact", {"d5h1"}),
    ("Capture Queen (Kxe2)", "4k3/8/8/8/8/8/4q3/4K3 w - - 0 1", "exact", {"e1e2"}),
    ("Avoid Stalemate & Mate", "7k/8/5K2/6Q1/8/8/8/8 w - - 0 1", "exact", {"g5g7", "g5h4"}),
    ("Refuse QxN #1 (Qd4xd5??)", "r1bqkb1r/ppp2ppp/2np4/3np3/2BQP3/2N2N2/PPPP1PPP/R1B1K2R w KQkq - 0 6", "forbid", {"d4d5"}),
    ("Refuse QxN #2 (Qa5xc3??)", "r1b1kbnr/pp1ppppp/2n5/q1p5/4P3/2NP1N2/PPP2PPP/R1BQKB1R b KQkq - 2 4", "forbid", {"a5c3"}),
    ("Scandinavian 7-Ply Qd5 Retreat (Refuse Qxf3??)", "rnb1kbnr/ppp1ppp1/8/3q3p/8/2N2N2/PPPP1PPP/R1BQKB1R b KQkq - 1 4", "exact", {"d5c5", "d5d8", "d5d6", "d5a5"}),
    ("Save Attacked Qd4 Safely", "r1bqkbnr/ppp1pppp/2n5/8/3Q4/5N2/PPP1PPPP/RNB1KB1R w KQkq - 2 4", "exact", {"d4e4", "d4e3", "d4d3", "d4d2", "d4d1", "d4a4", "d4c4", "d4f4", "d4g4", "d4h4"}),
    ("3-Ply Royal Fork (Nc7+ -> Nxa8!)", "r1b1kbnr/pp3ppp/2n1p3/1N1p2q1/8/8/PPPPPPPP/R1BQKBNR w KQkq - 0 5", "exact", {"b5c7"}),
    ("3-Ply Exchange Combo (Nxd5 exd5 Bxd5!)", "r2qkb1r/pppb1ppp/4p3/3n4/2B5/2N2N2/PPPP1PPP/R1BQ1RK1 w kq - 0 7", "exact", {"c3d5"}),
    ("Rescue Attacked Knight (d5 attacks Nc6!)", "r1bqkbnr/ppp2ppp/2n5/3Pp3/4P3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 4", "exact", {"c6e7", "c6b8", "c6a5", "c6d4"}),
    ("Fried Liver Defense (4.Ng5 -> 4...d5!, NOT h6??)", "r1bqkb1r/pppp1ppp/2n2n2/4p1N1/2B1P3/8/PPPP1PPP/RNBQK2R b KQkq - 5 4", "exact", {"d7d5", "f8c5"}),
    ("Opening: 1.e4 e5 2.Nf3 -> Defend e5 (Nc6/Nf6/d6)", "rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2", "exact", {"b8c6", "g8f6", "d7d6"}),
    ("Opening: Sound Reply to 1.d4", "rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq - 0 1", "exact", {"d7d5", "g8f6", "e7e6", "c7c5", "d7d6", "g7g6"}),
]

suite_boards = [chess.Board(f) for _, f, _, _ in SUITE_2PLY_Q3]
suite_res = select_move_2ply_q3_svg_batch(suite_boards, _exact_svg_eval_u8)
passed_cnt = 0
print("\n2-Ply + Ply-3 Quiescence Tactical, Anti-Hang & Opening Suite:")
for (name, _fen, mode, ref_set), (mv, sc, uci_sc, _) in zip(SUITE_2PLY_Q3, suite_res):
    if mode == "exact":
        ok = mv.uci() in ref_set
        detail = f"played {mv.uci()} (Q={sc}/255)"
    else:
        bad_u = next(iter(ref_set))
        ok = mv.uci() not in ref_set and sc > uci_sc.get(bad_u, 0) + 20
        detail = f"played {mv.uci()} (Q={sc}) vs forbidden {bad_u} (Q={uci_sc.get(bad_u)})"
    passed_cnt += int(ok)
    print(f"  [{'PASS' if ok else 'WARN'}] {name:50s} -> {detail}")
print(f"Suite Score: {passed_cnt}/{len(SUITE_2PLY_Q3)} passed")

# Export 24 Golden Positions strictly from `ai_color == bd.turn` test instances so `test_chess.py::test_01` matches 0-LSB!
stm_test_indices = [idx for idx, is_s in enumerate(is_stm_test) if is_s]
assert len(stm_test_indices) >= 24
golden_positions = []
for k, idx in enumerate(stm_test_indices[:24]):
    golden_positions.append({
        "id": f"chess_test_{k:02d}",
        "fen": fens_test[idx],
        "regime": regimes_test[idx],
        "stockfish_wdl": round(float(y_test_f[idx]), 5),
        "svg_u8": int(svg_u8_test[idx]),
        "svg_prob": round(float(svg_probs_test[idx]), 5),
        "svg_logit": round(float(svg_logits_test[idx]), 4),
    })

export_spec["metrics"] = {
    "val_wdl_mae_qat": round(float(best_metrics[0]), 5),
    "val_stockfish_corr_r": round(float(best_metrics[1]), 5),
    "val_sibling_rank_acc": round(float(best_metrics[2]), 5),
    "test_wdl_mae_svg_uint8": round(float(test_mae_svg), 5),
    "test_stockfish_corr_svg_uint8": round(float(test_corr_svg), 5),
    "qat_to_svg_median_lsb_diff": int(round(float(np.median(lsb_diff)))),
    "qat_to_svg_max_lsb_diff": int(round(float(lsb_diff.max()))),
    "qat_to_svg_max_logit_diff": round(float(logit_abs_diff.max()), 6),
    "active_lut_bins_of_256": int(active_lut_bins),
    "tactical_opening_suite_passed": f"{passed_cnt}/{len(SUITE_2PLY_Q3)}",
    "total_parameters": int(total_params),
    "num_labeled_positions": int(len(all_insts)),
}
export_spec["golden_positions"] = golden_positions

OUT_PATH = "weights.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(export_spec, f, separators=(",", ":"))

size_kb = os.path.getsize(OUT_PATH) / 1024.0
print(
    f"\nSaved {OUT_PATH} ({size_kb:.1f} KB, {total_params:,} params, M={export_spec['layers']['value_head']['logit_half_range_M']}) | "
    f"Test Stockfish r = {test_corr_svg:.4f} | Test MAE = {test_mae_svg:.4f}"
)

try:
    from google.colab import files
    files.download(OUT_PATH)
    print("Triggered browser download for weights.json!")
except ImportError:
    print("Not running inside Google Colab; weights.json written to current directory.")
